# 05 — Modeling

This notebook contains the historical target/feature preparation needed for model training plus baseline models, Random Forest, Gradient Boosting, tuning, validation and model comparison. It is extracted from the validated Phase 7–9 final notebook and preserves executed outputs.

In [ ]:
 # ============================================================
# PHASE 7: EXPLORATORY DATA ANALYSIS, RFM & CHURN ANALYSIS
# Meridian Retail Group Customer Churn Capstone
# ============================================================
#
# BUSINESS OBJECTIVE
# ------------------------------------------------------------
# Understand customer behaviour, purchasing patterns,
# customer segments, and support experience in order to
# investigate customer churn at Meridian Retail Group.
#
# PHASE 7 MANDATORY REQUIREMENTS
# ------------------------------------------------------------
# 1. Compute and interpret overall churn rate.
# 2. Compute and interpret churn by:
#       - Membership tier
#       - Preferred channel
#       - State
#       - Referral source
#       - Signup cohort
# 3. Engineer customer-level RFM features:
#       - Recency
#       - Frequency
#       - Monetary
# 4. Investigate support-ticket activity and its relationship
#    with churn:
#       - Ticket count
#       - Satisfaction score
#       - Resolution time
# 5. Perform univariate analysis.
# 6. Perform multivariate analysis.
# 7. Identify at least five concrete, non-obvious findings.
#
# LOCKED CHURN DEFINITION
# ------------------------------------------------------------
# Snapshot date: 2026-07-01
#
# A customer is CHURNED if they have NO order with
# status = 'Completed' during the preceding 180 days.
#
# This includes customers who have never ordered.
#
# Otherwise, the customer is ACTIVE.
#
# IMPORTANT:
# Completed orders occurring AFTER 2026-07-01 must not be
# used when determining churn status at the snapshot date.
#
# SOURCE OF TRUTH
# ------------------------------------------------------------
# The validated processed datasets created in Phase 6
# will be used for this analysis.
#
# Phase 6 final validation:
# Customers       = 6,090
# Products        =   220
# Orders          = 28,252
# Order Items     = 73,405
# Support Tickets =  5,200
#
# Phase 6 validation result: PASS
# ============================================================

print("PHASE 7: EDA, RFM & CHURN ANALYSIS")
print("=" * 80)
print("Meridian Retail Group")
print("=" * 80)

print("\nBusiness Objective:")
print(
    "Understand customer behaviour, purchasing patterns, "
    "support experience and churn characteristics."
)

print("\nChurn Snapshot Date: 2026-07-01")
print("Churn Window: 180 days")
print("Churn Definition: No completed order in preceding 180 days")

print("\nPhase 6 Source Data:")
print("- Customers:       6,090")
print("- Products:          220")
print("- Orders:         28,252")
print("- Order Items:    73,405")
print("- Support Tickets: 5,200")

print("\nPhase 6 Validation: PASS")
print("=" * 80)

PHASE 7: EDA, RFM & CHURN ANALYSIS
Meridian Retail Group

Business Objective:
Understand customer behaviour, purchasing patterns, support experience and churn characteristics.

Churn Snapshot Date: 2026-07-01
Churn Window: 180 days
Churn Definition: No completed order in preceding 180 days

Phase 6 Source Data:
- Customers:       6,090
- Products:          220
- Orders:         28,252
- Order Items:    73,405
- Support Tickets: 5,200

Phase 6 Validation: PASS


In [ ]:
# ============================================================
# CELL 2: IMPORT LIBRARIES AND SET PROJECT PATHS
# ============================================================
#
# Purpose:
# 1. Import the Python libraries required for Phase 7.
# 2. Mount Google Drive.
# 3. Define the project and processed-data paths.
#
# Libraries:
# - pandas: data loading, cleaning and analysis
# - numpy: numerical calculations
# - matplotlib: static visualizations
# - seaborn: statistical visualizations
# - plotly: interactive visualizations
#
# NOTE:
# The processed CSV files created and validated in Phase 6
# are the source of truth for this phase.
# ============================================================

# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from pathlib import Path


# -----------------------------
# 2. MOUNT GOOGLE DRIVE
# -----------------------------

from google.colab import drive

drive.mount('/content/drive')


# -----------------------------
# 3. DEFINE PROJECT PATHS
# -----------------------------

PROJECT_PATH = Path(
    "/content/drive/MyDrive/PORA Capstone"
)

PROCESSED_PATH = (
    PROJECT_PATH / "data" / "processed"
)


# -----------------------------
# 4. DISPLAY PATH INFORMATION
# -----------------------------

print("=" * 80)
print("PHASE 7 ENVIRONMENT SETUP")
print("=" * 80)

print("\nProject path:")
print(PROJECT_PATH)

print("\nProcessed data path:")
print(PROCESSED_PATH)

print("\nProcessed directory exists:",
      PROCESSED_PATH.exists())

print("Processed directory is a folder:",
      PROCESSED_PATH.is_dir())


# -----------------------------
# 5. DISPLAY LIBRARY VERSIONS
# -----------------------------

print("\nLIBRARY VERSIONS")
print("-" * 80)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seaborn:", sns.__version__)
print("Plotly:", px.__version__ if hasattr(px, "__version__") else "Installed")

Mounted at /content/drive
PHASE 7 ENVIRONMENT SETUP

Project path:
/content/drive/MyDrive/PORA Capstone

Processed data path:
/content/drive/MyDrive/PORA Capstone/data/processed

Processed directory exists: True
Processed directory is a folder: True

LIBRARY VERSIONS
--------------------------------------------------------------------------------
Pandas: 2.2.3
NumPy: 2.1.3
Matplotlib: 3.10.0
Seaborn: 0.13.2
Plotly: Installed


In [ ]:
# ============================================================
# CELL 3: VERIFY PROCESSED DATASET FILES
# ============================================================
#
# Purpose:
# Confirm that all five validated Phase 6 processed CSV files
# exist in the expected directory before analysis begins.
#
# Expected files:
# - customers_clean.csv
# - products_clean.csv
# - orders_clean.csv
# - order_items_clean.csv
# - support_tickets_clean.csv
#
# This prevents accidental use of raw or outdated datasets.
# ============================================================

print("=" * 80)
print("PHASE 7 PROCESSED DATASET FILE VERIFICATION")
print("=" * 80)

# Expected processed files
expected_files = {
    "Customers": "customers_clean.csv",
    "Products": "products_clean.csv",
    "Orders": "orders_clean.csv",
    "Order Items": "order_items_clean.csv",
    "Support Tickets": "support_tickets_clean.csv"
}

file_checks = []

for dataset_name, filename in expected_files.items():

    file_path = PROCESSED_PATH / filename

    exists = file_path.exists()

    file_checks.append(exists)

    print(
        f"{dataset_name:<20} | "
        f"{filename:<30} | "
        f"{'FOUND' if exists else 'MISSING'}"
    )

print("\n" + "-" * 80)

if all(file_checks):
    print("RESULT: PASS")
    print("All five validated processed datasets are available.")
else:
    print("RESULT: REVIEW REQUIRED")
    print("One or more processed datasets could not be found.")

print("=" * 80)

PHASE 7 PROCESSED DATASET FILE VERIFICATION
Customers            | customers_clean.csv            | FOUND
Products             | products_clean.csv             | FOUND
Orders               | orders_clean.csv               | FOUND
Order Items          | order_items_clean.csv          | FOUND
Support Tickets      | support_tickets_clean.csv      | FOUND

--------------------------------------------------------------------------------
RESULT: PASS
All five validated processed datasets are available.


In [ ]:
# ============================================================
# CELL 4: LOAD THE PROCESSED DATASETS
# ============================================================
#
# Purpose:
# Load the five validated Phase 6 datasets into Pandas
# DataFrames for Phase 7 analysis.
#
# Date columns are explicitly parsed during loading because
# CSV files do not preserve Pandas datetime dtypes.
#
# Date columns:
# - Customers: date_of_birth, signup_date
# - Products: launch_date
# - Orders: order_date
# - Support Tickets: ticket_date
# ============================================================

print("=" * 80)
print("LOADING PHASE 6 PROCESSED DATASETS")
print("=" * 80)


# ------------------------------------------------------------
# 1. CUSTOMERS
# ------------------------------------------------------------

customers = pd.read_csv(
    PROCESSED_PATH / "customers_clean.csv",
    parse_dates=["date_of_birth", "signup_date"]
)


# ------------------------------------------------------------
# 2. PRODUCTS
# ------------------------------------------------------------

products = pd.read_csv(
    PROCESSED_PATH / "products_clean.csv",
    parse_dates=["launch_date"]
)


# ------------------------------------------------------------
# 3. ORDERS
# ------------------------------------------------------------

orders = pd.read_csv(
    PROCESSED_PATH / "orders_clean.csv",
    parse_dates=["order_date"]
)


# ------------------------------------------------------------
# 4. ORDER ITEMS
# ------------------------------------------------------------

order_items = pd.read_csv(
    PROCESSED_PATH / "order_items_clean.csv"
)


# ------------------------------------------------------------
# 5. SUPPORT TICKETS
# ------------------------------------------------------------

support_tickets = pd.read_csv(
    PROCESSED_PATH / "support_tickets_clean.csv",
    parse_dates=["ticket_date"]
)


# ------------------------------------------------------------
# 6. DISPLAY LOADED DATASET SHAPES
# ------------------------------------------------------------

print("\nDATASET SHAPES")
print("-" * 80)

print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Support Tickets:", support_tickets.shape)


# ------------------------------------------------------------
# 7. CONFIRM DATE COLUMN TYPES
# ------------------------------------------------------------

print("\nDATE COLUMN DATA TYPES")
print("-" * 80)

print("customers.date_of_birth:",
      customers["date_of_birth"].dtype)

print("customers.signup_date:",
      customers["signup_date"].dtype)

print("products.launch_date:",
      products["launch_date"].dtype)

print("orders.order_date:",
      orders["order_date"].dtype)

print("support_tickets.ticket_date:",
      support_tickets["ticket_date"].dtype)


# ------------------------------------------------------------
# 8. BASIC LOAD VALIDATION
# ------------------------------------------------------------

expected_shapes = {
    "Customers": (6090, 14),
    "Products": (220, 9),
    "Orders": (28252, 9),
    "Order Items": (73405, 5),
    "Support Tickets": (5200, 8)
}

actual_shapes = {
    "Customers": customers.shape,
    "Products": products.shape,
    "Orders": orders.shape,
    "Order Items": order_items.shape,
    "Support Tickets": support_tickets.shape
}

print("\nSHAPE VALIDATION")
print("-" * 80)

load_checks = []

for dataset_name in expected_shapes:

    passed = (
        actual_shapes[dataset_name]
        == expected_shapes[dataset_name]
    )

    load_checks.append(passed)

    print(
        f"{dataset_name:<20} | "
        f"{'PASS' if passed else 'FAIL'} | "
        f"Expected: {expected_shapes[dataset_name]} | "
        f"Actual: {actual_shapes[dataset_name]}"
    )


# ------------------------------------------------------------
# 9. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 80)

if all(load_checks):
    print("LOAD RESULT: PASS")
    print("All five processed datasets loaded successfully.")
else:
    print("LOAD RESULT: REVIEW REQUIRED")

print("=" * 80)

LOADING PHASE 6 PROCESSED DATASETS

DATASET SHAPES
--------------------------------------------------------------------------------
Customers: (6090, 14)
Products: (220, 9)
Orders: (28252, 9)
Order Items: (73405, 5)
Support Tickets: (5200, 8)

DATE COLUMN DATA TYPES
--------------------------------------------------------------------------------
customers.date_of_birth: datetime64[ns]
customers.signup_date: datetime64[ns]
products.launch_date: datetime64[ns]
orders.order_date: datetime64[ns]
support_tickets.ticket_date: datetime64[ns]

SHAPE VALIDATION
--------------------------------------------------------------------------------
Customers            | PASS | Expected: (6090, 14) | Actual: (6090, 14)
Products             | PASS | Expected: (220, 9) | Actual: (220, 9)
Orders               | PASS | Expected: (28252, 9) | Actual: (28252, 9)
Order Items          | PASS | Expected: (73405, 5) | Actual: (73405, 5)
Support Tickets      | PASS | Expected: (5200, 8) | Actual: (5200, 8)

LOAD 

In [ ]:
# ============================================================
# CELL 35: ML TARGET DESIGN & LEAKAGE-SAFE PREDICTION FRAMEWORK
# Meridian Retail Group - Customer Churn Project
# ============================================================

print("=" * 70)
print("CELL 35: MACHINE LEARNING TARGET DESIGN")
print("=" * 70)

# ------------------------------------------------------------
# 1. IMPORTANT DISTINCTION
# ------------------------------------------------------------
# The official EDA/business churn definition remains LOCKED:
#
# Snapshot date: 2026-07-01
# Churn = no Completed order in the preceding 180 days.
#
# We will NOT replace or modify that definition.
#
# For Machine Learning, however, we need to predict FUTURE churn.
# Therefore, the model will learn from historical prediction dates
# where the future outcome is already known.

ML_SCORING_DATE = SNAPSHOT_DATE
ML_HORIZON_DAYS = 180

# The ML prediction target will be:
#
# ml_target = 1
#     if the customer has NO Completed order during the
#     180 days AFTER the prediction date.
#
# ml_target = 0
#     if the customer has at least one Completed order during
#     that future 180-day period.
#
# This makes the target genuinely forward-looking and prevents
# us from using the same information that defines current churn.

print("\nOFFICIAL BUSINESS / EDA CHURN DEFINITION")
print("-" * 50)
print(f"Snapshot date: {SNAPSHOT_DATE.date()}")
print(f"Lookback window: {CHURN_WINDOW_DAYS} days")
print("Churn = No Completed order in the preceding 180 days.")

print("\nMACHINE LEARNING PREDICTION TARGET")
print("-" * 50)
print(f"Prediction horizon: {ML_HORIZON_DAYS} days")
print("Target = Future 180-day churn")
print("1 = No Completed order in the following 180 days")
print("0 = At least one Completed order in the following 180 days")

# ------------------------------------------------------------
# 2. CHECK THE AVAILABLE ORDER DATE RANGE
# ------------------------------------------------------------
completed_orders = orders[
    orders["order_status"].eq("Completed")
].copy()

completed_orders = completed_orders.dropna(
    subset=["order_date", "customer_id"]
)

latest_completed_date = completed_orders["order_date"].max()
earliest_completed_date = completed_orders["order_date"].min()

print("\nCOMPLETED ORDER DATE RANGE")
print("-" * 50)
print(f"Earliest Completed order: {earliest_completed_date.date()}")
print(f"Latest Completed order:   {latest_completed_date.date()}")

# ------------------------------------------------------------
# 3. DETERMINE THE LATEST DATE WITH A FULL 180-DAY
#    FUTURE OUTCOME AVAILABLE
# ------------------------------------------------------------
latest_training_cutoff = (
    latest_completed_date -
    pd.Timedelta(days=ML_HORIZON_DAYS)
)

print("\nLEAKAGE-SAFE TRAINING DATE LIMIT")
print("-" * 50)
print(
    f"Latest possible historical prediction date with a full "
    f"{ML_HORIZON_DAYS}-day outcome: "
    f"{latest_training_cutoff.date()}"
)

# ------------------------------------------------------------
# 4. CURRENT SCORING POPULATION
# ------------------------------------------------------------
# The current EDA already identified customers who are Active
# at the official 2026-07-01 snapshot.
#
# These customers will eventually receive predicted probabilities
# of becoming churned during the NEXT 180 days.
#
# Their future outcome is not yet available in our dataset,
# so they are a scoring population rather than a labelled test set.

current_active_customers = customers_eda.loc[
    customers_eda["churn_status"].eq("Active"),
    "customer_id"
].unique()

print("\nCURRENT SCORING POPULATION")
print("-" * 50)
print(
    f"Active customers at {ML_SCORING_DATE.date()}: "
    f"{len(current_active_customers):,}"
)

# ------------------------------------------------------------
# 5. LEAKAGE CONTROL PRINCIPLE
# ------------------------------------------------------------
print("\nLEAKAGE CONTROL PRINCIPLE")
print("-" * 50)
print(
    "For every historical prediction date, features will use only "
    "information available ON or BEFORE that date."
)

print(
    "The ML target will use only Completed orders occurring AFTER "
    "the prediction date and within the following 180 days."
)

print(
    "This prevents future purchasing behaviour from leaking into "
    "the predictors."
)

# ------------------------------------------------------------
# 6. FINAL DESIGN CHECK
# ------------------------------------------------------------
assert ML_HORIZON_DAYS == 180
assert latest_completed_date > earliest_completed_date
assert len(current_active_customers) == 2018

print("\n" + "=" * 70)
print("CELL 35 RESULT: ML TARGET FRAMEWORK DEFINED")
print("=" * 70)

print("✓ Official 2026-07-01 EDA churn definition remains unchanged")
print("✓ ML target is forward-looking")
print("✓ Prediction horizon = 180 days")
print("✓ Historical training dates must have a complete future window")
print("✓ Current active customers will be the eventual scoring population")
print("✓ Leakage-control rule established")
print("=" * 70)

CELL 35: MACHINE LEARNING TARGET DESIGN

OFFICIAL BUSINESS / EDA CHURN DEFINITION
--------------------------------------------------
Snapshot date: 2026-07-01
Lookback window: 180 days
Churn = No Completed order in the preceding 180 days.

MACHINE LEARNING PREDICTION TARGET
--------------------------------------------------
Prediction horizon: 180 days
Target = Future 180-day churn
1 = No Completed order in the following 180 days
0 = At least one Completed order in the following 180 days

COMPLETED ORDER DATE RANGE
--------------------------------------------------
Earliest Completed order: 2020-09-15
Latest Completed order:   2026-07-03

LEAKAGE-SAFE TRAINING DATE LIMIT
--------------------------------------------------
Latest possible historical prediction date with a full 180-day outcome: 2026-01-04

CURRENT SCORING POPULATION
--------------------------------------------------
Active customers at 2026-07-01: 2,018

LEAKAGE CONTROL PRINCIPLE
------------------------------------------

In [ ]:
# ============================================================
# CELL 36: DEFINE HISTORICAL ML PREDICTION DATES
# ============================================================

print("=" * 70)
print("CELL 36: HISTORICAL ML PREDICTION DATES")
print("=" * 70)

# ------------------------------------------------------------
# 1. WHY USE MULTIPLE HISTORICAL PREDICTION DATES?
# ------------------------------------------------------------
# A single prediction date would give us only one snapshot
# of customer behaviour.
#
# Instead, we create repeated historical snapshots.
#
# At each prediction date:
#   - Features use information available up to that date.
#   - The next 180 days determine the ML target.
#
# This creates a more realistic training dataset while
# preserving temporal ordering and preventing leakage.

ML_FIRST_PREDICTION_DATE = pd.Timestamp("2021-07-01")

# We cannot use dates later than the latest date for which
# a complete 180-day future outcome is available.
ML_LAST_PREDICTION_DATE = latest_training_cutoff

# Generate monthly prediction dates.
ML_PREDICTION_DATES = pd.date_range(
    start=ML_FIRST_PREDICTION_DATE,
    end=ML_LAST_PREDICTION_DATE,
    freq="MS"
)

print("\nPREDICTION DATE RANGE")
print("-" * 50)
print(f"First prediction date: {ML_PREDICTION_DATES.min().date()}")
print(f"Last prediction date:  {ML_PREDICTION_DATES.max().date()}")
print(f"Number of prediction dates: {len(ML_PREDICTION_DATES):,}")

# ------------------------------------------------------------
# 2. CHECK THAT EVERY PREDICTION DATE HAS A FULL 180-DAY
#    FUTURE OUTCOME WINDOW
# ------------------------------------------------------------

future_end_dates = (
    ML_PREDICTION_DATES +
    pd.Timedelta(days=ML_HORIZON_DAYS)
)

prediction_date_check = pd.DataFrame({
    "prediction_date": ML_PREDICTION_DATES,
    "future_end_date": future_end_dates
})

prediction_date_check["has_complete_future_window"] = (
    prediction_date_check["future_end_date"]
    <= latest_completed_date
)

print("\nFUTURE OUTCOME WINDOW CHECK")
print("-" * 50)

print(
    prediction_date_check[
        "has_complete_future_window"
    ].value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 3. RETAIN ONLY FULLY OBSERVABLE TRAINING WINDOWS
# ------------------------------------------------------------

ML_PREDICTION_DATES = prediction_date_check.loc[
    prediction_date_check["has_complete_future_window"],
    "prediction_date"
].tolist()

print("\nFINAL HISTORICAL PREDICTION DATES")
print("-" * 50)
print(f"Usable prediction dates: {len(ML_PREDICTION_DATES):,}")
print(f"First usable date: {ML_PREDICTION_DATES[0].date()}")
print(f"Last usable date:  {ML_PREDICTION_DATES[-1].date()}")

# ------------------------------------------------------------
# 4. DISPLAY THE FIRST AND LAST FEW DATES
# ------------------------------------------------------------

print("\nFIRST 5 PREDICTION DATES")
print("-" * 50)

for date in ML_PREDICTION_DATES[:5]:
    print(date.date())

print("\nLAST 5 PREDICTION DATES")
print("-" * 50)

for date in ML_PREDICTION_DATES[-5:]:
    print(date.date())

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

assert len(ML_PREDICTION_DATES) > 0
assert all(
    date + pd.Timedelta(days=ML_HORIZON_DAYS)
    <= latest_completed_date
    for date in ML_PREDICTION_DATES
)

assert ML_PREDICTION_DATES[-1] <= ML_LAST_PREDICTION_DATE

print("\n" + "=" * 70)
print("CELL 36 RESULT: HISTORICAL PREDICTION DATES DEFINED")
print("=" * 70)

print("✓ Multiple historical snapshots will be used")
print("✓ Prediction dates are monthly")
print("✓ Every training date has a complete 180-day future window")
print("✓ Future outcome data will be available for every training snapshot")
print("✓ Temporal leakage control remains in place")
print("=" * 70)

CELL 36: HISTORICAL ML PREDICTION DATES

PREDICTION DATE RANGE
--------------------------------------------------
First prediction date: 2021-07-01
Last prediction date:  2026-01-01
Number of prediction dates: 55

FUTURE OUTCOME WINDOW CHECK
--------------------------------------------------
has_complete_future_window
True    55
Name: count, dtype: int64

FINAL HISTORICAL PREDICTION DATES
--------------------------------------------------
Usable prediction dates: 55
First usable date: 2021-07-01
Last usable date:  2026-01-01

FIRST 5 PREDICTION DATES
--------------------------------------------------
2021-07-01
2021-08-01
2021-09-01
2021-10-01
2021-11-01

LAST 5 PREDICTION DATES
--------------------------------------------------
2025-09-01
2025-10-01
2025-11-01
2025-12-01
2026-01-01

CELL 36 RESULT: HISTORICAL PREDICTION DATES DEFINED
✓ Multiple historical snapshots will be used
✓ Prediction dates are monthly
✓ Every training date has a complete 180-day future window
✓ Future outcome d

In [ ]:
# ============================================================
# CELL 37: BUILD AND VALIDATE HISTORICAL ML TARGET
# ============================================================

print("=" * 70)
print("CELL 37: HISTORICAL FUTURE-CHURN TARGET")
print("=" * 70)

# ------------------------------------------------------------
# 1. PREPARE COMPLETED ORDERS FOR FUTURE-LABEL CALCULATION
# ------------------------------------------------------------
# Only Completed orders can count as evidence that a customer
# remained active.
#
# Orders that are Returned, Cancelled or Pending do not count.

completed_orders_ml = orders[
    orders["order_status"].eq("Completed")
].copy()

completed_orders_ml = completed_orders_ml.dropna(
    subset=["customer_id", "order_date"]
)

# Keep only the columns required for target construction.
completed_orders_ml = completed_orders_ml[
    ["customer_id", "order_date"]
].copy()

print("\nCOMPLETED ORDERS AVAILABLE FOR ML TARGET")
print("-" * 50)
print(f"Rows: {len(completed_orders_ml):,}")
print(
    f"Unique customers: "
    f"{completed_orders_ml['customer_id'].nunique():,}"
)
print(
    f"Date range: "
    f"{completed_orders_ml['order_date'].min().date()} "
    f"to "
    f"{completed_orders_ml['order_date'].max().date()}"
)

# ------------------------------------------------------------
# 2. CREATE CUSTOMER-PREDICTION-DATE BASE
# ------------------------------------------------------------
# A customer can appear at multiple prediction dates.
#
# However, we should only include a customer at a prediction
# date if the customer had already signed up by that date.
#
# This prevents us from creating artificial observations for
# customers who did not yet exist.

customer_base = customers[
    ["customer_id", "signup_date"]
].copy()

ml_target_rows = []

for prediction_date in ML_PREDICTION_DATES:

    # Customers who existed by the prediction date.
    eligible_customers = customer_base[
        customer_base["signup_date"] <= prediction_date
    ][["customer_id"]].copy()

    # Define the future 180-day window.
    future_start = prediction_date
    future_end = (
        prediction_date +
        pd.Timedelta(days=ML_HORIZON_DAYS)
    )

    # Completed orders strictly AFTER the prediction date
    # and ON/BEFORE the end of the future window.
    future_orders = completed_orders_ml[
        (completed_orders_ml["order_date"] > future_start) &
        (completed_orders_ml["order_date"] <= future_end)
    ]

    # Customers who made at least one Completed purchase
    # during the future window.
    future_purchasers = set(
        future_orders["customer_id"].unique()
    )

    # Create the target.
    eligible_customers["prediction_date"] = prediction_date

    eligible_customers["ml_target"] = (
        ~eligible_customers["customer_id"].isin(
            future_purchasers
        )
    ).astype(int)

    ml_target_rows.append(eligible_customers)

# Combine all historical prediction snapshots.
ml_target_df = pd.concat(
    ml_target_rows,
    ignore_index=True
)

# ------------------------------------------------------------
# 3. BASIC TARGET DATASET CHECKS
# ------------------------------------------------------------

print("\nHISTORICAL ML TARGET DATASET")
print("-" * 50)
print(f"Rows: {len(ml_target_df):,}")
print(
    f"Unique customers: "
    f"{ml_target_df['customer_id'].nunique():,}"
)
print(
    f"Prediction dates: "
    f"{ml_target_df['prediction_date'].nunique():,}"
)

# ------------------------------------------------------------
# 4. TARGET DISTRIBUTION
# ------------------------------------------------------------

target_distribution = (
    ml_target_df["ml_target"]
    .value_counts()
    .sort_index()
)

print("\nOVERALL ML TARGET DISTRIBUTION")
print("-" * 50)

for target_value, count in target_distribution.items():

    percentage = (
        count /
        len(ml_target_df) *
        100
    )

    label = (
        "No future Completed order (Churn)"
        if target_value == 1
        else
        "Future Completed order (Active)"
    )

    print(
        f"{target_value} = {label}: "
        f"{count:,} ({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# 5. TARGET DISTRIBUTION BY PREDICTION DATE
# ------------------------------------------------------------

target_by_date = (
    ml_target_df
    .groupby("prediction_date")["ml_target"]
    .agg(
        customers="count",
        churned="sum",
        churn_rate="mean"
    )
    .reset_index()
)

target_by_date["churn_rate_pct"] = (
    target_by_date["churn_rate"] * 100
)

print("\nTARGET DISTRIBUTION BY PREDICTION DATE")
print("-" * 50)

print(
    target_by_date[
        [
            "prediction_date",
            "customers",
            "churned",
            "churn_rate_pct"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. CHECK FOR INVALID TARGET VALUES
# ------------------------------------------------------------

invalid_target_count = (
    ~ml_target_df["ml_target"].isin([0, 1])
).sum()

missing_target_count = (
    ml_target_df["ml_target"].isna().sum()
)

print("\nTARGET VALIDATION")
print("-" * 50)
print(f"Missing target values: {missing_target_count}")
print(f"Invalid target values: {invalid_target_count}")

# ------------------------------------------------------------
# 7. CHECK FOR DUPLICATE CUSTOMER-PREDICTION OBSERVATIONS
# ------------------------------------------------------------

duplicate_observations = (
    ml_target_df
    .duplicated(
        subset=["customer_id", "prediction_date"]
    )
    .sum()
)

print(
    f"Duplicate customer-prediction observations: "
    f"{duplicate_observations}"
)

# ------------------------------------------------------------
# 8. VALIDATION ASSERTIONS
# ------------------------------------------------------------

assert len(ml_target_df) > 0
assert ml_target_df["ml_target"].isin([0, 1]).all()
assert ml_target_df["ml_target"].notna().all()

assert duplicate_observations == 0

assert (
    ml_target_df["prediction_date"]
    .nunique()
    == len(ML_PREDICTION_DATES)
)

# Every customer-prediction observation must correspond
# to a customer who had already signed up.
signup_check = ml_target_df.merge(
    customer_base,
    on="customer_id",
    how="left"
)

assert (
    signup_check["signup_date"]
    <= signup_check["prediction_date"]
).all()

print("\n" + "=" * 70)
print("CELL 37 RESULT: HISTORICAL ML TARGET CREATED")
print("=" * 70)

print("✓ Future-churn target created")
print("✓ Target uses only Completed orders")
print("✓ Future window = 180 days")
print("✓ Customers only enter after signup")
print("✓ Target contains only 0 and 1")
print("✓ No missing target values")
print("✓ No duplicate customer-prediction observations")
print("✓ All 55 prediction dates represented")
print("=" * 70)

CELL 37: HISTORICAL FUTURE-CHURN TARGET

COMPLETED ORDERS AVAILABLE FOR ML TARGET
--------------------------------------------------
Rows: 25,079
Unique customers: 5,643
Date range: 2020-09-15 to 2026-07-03

HISTORICAL ML TARGET DATASET
--------------------------------------------------
Rows: 175,099
Unique customers: 5,564
Prediction dates: 55

OVERALL ML TARGET DISTRIBUTION
--------------------------------------------------
0 = Future Completed order (Active): 66,269 (37.85%)
1 = No future Completed order (Churn): 108,830 (62.15%)

TARGET DISTRIBUTION BY PREDICTION DATE
--------------------------------------------------
prediction_date  customers  churned  churn_rate_pct
     2021-07-01        840      590       70.238095
     2021-08-01        923      635       68.797400
     2021-09-01       1026      714       69.590643
     2021-10-01       1108      766       69.133574
     2021-11-01       1182      804       68.020305
     2021-12-01       1264      863       68.275316
     2

In [ ]:
# ============================================================
# CELL 38: INDEPENDENT VALIDATION OF HISTORICAL ML TARGET
# ============================================================

print("=" * 70)
print("CELL 38: INDEPENDENT ML TARGET VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. RECONSTRUCT FUTURE PURCHASE COUNTS
# ------------------------------------------------------------
# This independently counts Completed orders in each customer's
# future 180-day window.
#
# We use the already-created ml_target_df as the customer/date
# population, but independently calculate the future purchase
# count from the orders data.

validation_rows = []

for prediction_date in ML_PREDICTION_DATES:

    future_end = (
        prediction_date +
        pd.Timedelta(days=ML_HORIZON_DAYS)
    )

    future_orders = completed_orders_ml[
        (completed_orders_ml["order_date"] > prediction_date) &
        (completed_orders_ml["order_date"] <= future_end)
    ]

    future_counts = (
        future_orders
        .groupby("customer_id")
        .size()
        .rename("future_completed_orders")
    )

    date_population = ml_target_df[
        ml_target_df["prediction_date"] == prediction_date
    ][
        ["customer_id", "prediction_date", "ml_target"]
    ].copy()

    date_population = date_population.merge(
        future_counts,
        on="customer_id",
        how="left"
    )

    date_population["future_completed_orders"] = (
        date_population["future_completed_orders"]
        .fillna(0)
        .astype(int)
    )

    validation_rows.append(date_population)

target_validation = pd.concat(
    validation_rows,
    ignore_index=True
)

# ------------------------------------------------------------
# 2. CHECK TARGET = 0
# ------------------------------------------------------------
# Target 0 means the customer must have at least one
# Completed order in the future 180-day window.

target_zero_errors = (
    (target_validation["ml_target"] == 0) &
    (target_validation["future_completed_orders"] == 0)
).sum()

# ------------------------------------------------------------
# 3. CHECK TARGET = 1
# ------------------------------------------------------------
# Target 1 means the customer must have ZERO Completed orders
# in the future 180-day window.

target_one_errors = (
    (target_validation["ml_target"] == 1) &
    (target_validation["future_completed_orders"] > 0)
).sum()

print("\nTARGET LOGIC VALIDATION")
print("-" * 50)
print(
    "Target 0 observations incorrectly having "
    f"zero future Completed orders: {target_zero_errors}"
)
print(
    "Target 1 observations incorrectly having "
    f"future Completed orders: {target_one_errors}"
)

# ------------------------------------------------------------
# 4. CHECK FUTURE PURCHASE COUNTS BY TARGET
# ------------------------------------------------------------

target_purchase_summary = (
    target_validation
    .groupby("ml_target")["future_completed_orders"]
    .agg(
        observations="count",
        minimum="min",
        median="median",
        maximum="max"
    )
    .reset_index()
)

print("\nFUTURE COMPLETED ORDERS BY ML TARGET")
print("-" * 50)
print(target_purchase_summary.to_string(index=False))

# ------------------------------------------------------------
# 5. INSPECT SELECTED PREDICTION DATES
# ------------------------------------------------------------

selected_dates = [
    ML_PREDICTION_DATES[0],
    ML_PREDICTION_DATES[len(ML_PREDICTION_DATES) // 2],
    ML_PREDICTION_DATES[-1]
]

print("\nSELECTED PREDICTION-DATE CHECK")
print("-" * 50)

for date in selected_dates:

    subset = target_validation[
        target_validation["prediction_date"] == date
    ]

    target_0 = subset[
        subset["ml_target"] == 0
    ]

    target_1 = subset[
        subset["ml_target"] == 1
    ]

    print(f"\nPrediction date: {date.date()}")

    print(
        f"  Target 0 customers: {len(target_0):,}"
    )
    print(
        f"  Target 0 with future purchase: "
        f"{(target_0['future_completed_orders'] > 0).sum():,}"
    )

    print(
        f"  Target 1 customers: {len(target_1):,}"
    )
    print(
        f"  Target 1 with no future purchase: "
        f"{(target_1['future_completed_orders'] == 0).sum():,}"
    )

# ------------------------------------------------------------
# 6. CHECK FOR NEGATIVE FUTURE PURCHASE COUNTS
# ------------------------------------------------------------

negative_future_counts = (
    target_validation["future_completed_orders"] < 0
).sum()

print("\nADDITIONAL VALIDATION")
print("-" * 50)
print(
    f"Negative future purchase counts: "
    f"{negative_future_counts}"
)

# ------------------------------------------------------------
# 7. FINAL ASSERTIONS
# ------------------------------------------------------------

assert target_zero_errors == 0
assert target_one_errors == 0
assert negative_future_counts == 0

assert (
    target_validation["future_completed_orders"].min()
    >= 0
)

# Target 0 must always have at least one future purchase.
assert (
    target_validation.loc[
        target_validation["ml_target"] == 0,
        "future_completed_orders"
    ] > 0
).all()

# Target 1 must always have zero future purchases.
assert (
    target_validation.loc[
        target_validation["ml_target"] == 1,
        "future_completed_orders"
    ] == 0
).all()

print("\n" + "=" * 70)
print("CELL 38 RESULT: ML TARGET VALIDATION PASSED")
print("=" * 70)

print("✓ Every target 0 customer has a future Completed order")
print("✓ Every target 1 customer has no future Completed order")
print("✓ Future purchase counts are logically valid")
print("✓ Selected historical dates behave correctly")
print("✓ No target-construction contradictions detected")
print("=" * 70)

CELL 38: INDEPENDENT ML TARGET VALIDATION

TARGET LOGIC VALIDATION
--------------------------------------------------
Target 0 observations incorrectly having zero future Completed orders: 0
Target 1 observations incorrectly having future Completed orders: 0

FUTURE COMPLETED ORDERS BY ML TARGET
--------------------------------------------------
 ml_target  observations  minimum  median  maximum
         0         66269        1     1.0       10
         1        108830        0     0.0        0

SELECTED PREDICTION-DATE CHECK
--------------------------------------------------

Prediction date: 2021-07-01
  Target 0 customers: 250
  Target 0 with future purchase: 250
  Target 1 customers: 590
  Target 1 with no future purchase: 590

Prediction date: 2023-10-01
  Target 0 customers: 1,284
  Target 0 with future purchase: 1,284
  Target 1 customers: 1,880
  Target 1 with no future purchase: 1,880

Prediction date: 2026-01-01
  Target 0 customers: 1,568
  Target 0 with future purchase: 1,

In [ ]:
# ============================================================
# CELL 39: DEFINE LEAKAGE-SAFE ML FEATURE RULES
# ============================================================

print("=" * 70)
print("CELL 39: ML FEATURE-ENGINEERING RULES")
print("=" * 70)

# ------------------------------------------------------------
# 1. CUSTOMER ATTRIBUTES
# ------------------------------------------------------------
# These are customer characteristics available from the
# customer table.
#
# They will be treated as predictors because they describe
# the customer at the prediction point.

CUSTOMER_FEATURES = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in"
]

# ------------------------------------------------------------
# 2. HISTORICAL PURCHASING FEATURES
# ------------------------------------------------------------
# These features will be calculated using ONLY Completed orders
# whose order_date is ON or BEFORE the prediction date.

PURCHASE_FEATURES = [
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average"
]

# ------------------------------------------------------------
# 3. SUPPORT FEATURES
# ------------------------------------------------------------
# Support information will also be restricted to tickets that
# existed ON or BEFORE the prediction date.

SUPPORT_FEATURES = [
    "support_ticket_count",
    "satisfaction_average",
    "resolution_time_average"
]

# ------------------------------------------------------------
# 4. FEATURE SET
# ------------------------------------------------------------

ALL_ML_FEATURES = (
    CUSTOMER_FEATURES
    + PURCHASE_FEATURES
    + SUPPORT_FEATURES
)

print("\nCUSTOMER FEATURES")
print("-" * 50)
for feature in CUSTOMER_FEATURES:
    print(f"✓ {feature}")

print("\nPURCHASING FEATURES")
print("-" * 50)
for feature in PURCHASE_FEATURES:
    print(f"✓ {feature}")

print("\nSUPPORT FEATURES")
print("-" * 50)
for feature in SUPPORT_FEATURES:
    print(f"✓ {feature}")

# ------------------------------------------------------------
# 5. LEAKAGE RULES
# ------------------------------------------------------------

LEAKAGE_RULES = [
    "Features must use information available on or before the prediction date.",
    "Completed orders after the prediction date must not enter RFM features.",
    "Support tickets after the prediction date must not enter support features.",
    "The future 180-day purchase window is used only to create ml_target.",
    "No future order amount, future channel, or future support information may be used.",
    "The official 2026-07-01 EDA churn_flag will not be used as an ML predictor."
]

print("\nLEAKAGE-CONTROL RULES")
print("-" * 50)

for number, rule in enumerate(LEAKAGE_RULES, start=1):
    print(f"{number}. {rule}")

# ------------------------------------------------------------
# 6. IMPORTANT RFM INTERPRETATION
# ------------------------------------------------------------

print("\nRFM INTERPRETATION FOR ML")
print("-" * 50)
print(
    "Recency = days since the customer's latest Completed order "
    "known at the prediction date."
)

print(
    "Frequency = number of Completed orders known at the "
    "prediction date."
)

print(
    "Monetary = historical Completed-order spending known at "
    "the prediction date."
)

print(
    "These are historical predictors, not the future outcome."
)

# ------------------------------------------------------------
# 7. FEATURE/TARGET SEPARATION
# ------------------------------------------------------------

ML_TARGET_COLUMN = "ml_target"

print("\nFEATURE / TARGET SEPARATION")
print("-" * 50)
print(f"Target column: {ML_TARGET_COLUMN}")
print(f"Number of predictor features: {len(ALL_ML_FEATURES)}")

assert ML_TARGET_COLUMN not in ALL_ML_FEATURES
assert len(ALL_ML_FEATURES) == (
    len(CUSTOMER_FEATURES)
    + len(PURCHASE_FEATURES)
    + len(SUPPORT_FEATURES)
)

print("\n" + "=" * 70)
print("CELL 39 RESULT: FEATURE RULES DEFINED")
print("=" * 70)

print("✓ Customer attributes identified")
print("✓ Historical purchase features identified")
print("✓ Historical support features identified")
print("✓ Leakage rules explicitly documented")
print("✓ Target separated from predictors")
print("✓ RFM interpretation established")
print("=" * 70)

CELL 39: ML FEATURE-ENGINEERING RULES

CUSTOMER FEATURES
--------------------------------------------------
✓ membership_tier
✓ preferred_channel
✓ state
✓ referral_source
✓ marketing_opt_in

PURCHASING FEATURES
--------------------------------------------------
✓ recency_days
✓ frequency
✓ monetary_total
✓ monetary_average

SUPPORT FEATURES
--------------------------------------------------
✓ support_ticket_count
✓ satisfaction_average
✓ resolution_time_average

LEAKAGE-CONTROL RULES
--------------------------------------------------
1. Features must use information available on or before the prediction date.
2. Completed orders after the prediction date must not enter RFM features.
3. Support tickets after the prediction date must not enter support features.
4. The future 180-day purchase window is used only to create ml_target.
5. No future order amount, future channel, or future support information may be used.
6. The official 2026-07-01 EDA churn_flag will not be used as an ML pre

In [ ]:
# ============================================================
# CELL 40: BUILD HISTORICAL PURCHASE / RFM FEATURES
# ============================================================

print("=" * 70)
print("CELL 40: HISTORICAL PURCHASE / RFM FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 1. PREPARE CUSTOMER BASE
# ------------------------------------------------------------
# Every customer-prediction-date observation from Cell 37
# will receive purchasing features.

rfm_base = ml_target_df[
    ["customer_id", "prediction_date", "ml_target"]
].copy()

# ------------------------------------------------------------
# 2. PREPARE COMPLETED ORDERS
# ------------------------------------------------------------
# Only Completed orders are valid for RFM.
#
# We also keep order_id because Frequency should count
# unique completed orders.

rfm_orders = orders[
    orders["order_status"].eq("Completed")
].copy()

rfm_orders = rfm_orders.dropna(
    subset=["customer_id", "order_date"]
)

rfm_orders = rfm_orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "total_amount"
    ]
].copy()

# ------------------------------------------------------------
# 3. BUILD RFM FEATURES FOR EACH PREDICTION DATE
# ------------------------------------------------------------

rfm_feature_rows = []

for prediction_date in ML_PREDICTION_DATES:

    # Only historical orders known at this prediction date.
    historical_orders = rfm_orders[
        rfm_orders["order_date"] <= prediction_date
    ].copy()

    # Customer-level historical RFM.
    rfm_snapshot = (
        historical_orders
        .groupby("customer_id")
        .agg(
            last_completed_order=("order_date", "max"),
            frequency=("order_id", "nunique"),
            monetary_total=("total_amount", "sum"),
            monetary_average=("total_amount", "mean")
        )
        .reset_index()
    )

    # Add the prediction date.
    rfm_snapshot["prediction_date"] = prediction_date

    # Calculate recency only for customers with completed
    # purchasing history.
    rfm_snapshot["recency_days"] = (
        prediction_date -
        rfm_snapshot["last_completed_order"]
    ).dt.days

    # We no longer need the last order date in the feature set.
    rfm_snapshot = rfm_snapshot.drop(
        columns=["last_completed_order"]
    )

    rfm_feature_rows.append(rfm_snapshot)

# Combine all historical RFM snapshots.
historical_rfm = pd.concat(
    rfm_feature_rows,
    ignore_index=True
)

# ------------------------------------------------------------
# 4. MERGE RFM FEATURES BACK TO THE FULL CUSTOMER-DATE BASE
# ------------------------------------------------------------
# This is important because customers who had not yet made
# a Completed order by a prediction date will not appear in
# historical_rfm.
#
# They must remain in the ML dataset.

ml_features = rfm_base.merge(
    historical_rfm,
    on=["customer_id", "prediction_date"],
    how="left"
)

# ------------------------------------------------------------
# 5. DEFINE ZERO-PURCHASE VALUES
# ------------------------------------------------------------
# No completed order history means:
#   Frequency = 0
#   Monetary total = 0
#
# Recency and monetary average remain missing because there
# is no meaningful value for those measurements.

ml_features["frequency"] = (
    ml_features["frequency"]
    .fillna(0)
    .astype(int)
)

ml_features["monetary_total"] = (
    ml_features["monetary_total"]
    .fillna(0)
)

# ------------------------------------------------------------
# 6. BASIC RFM VALIDATION
# ------------------------------------------------------------

print("\nHISTORICAL RFM DATASET")
print("-" * 50)

print(f"Rows: {len(ml_features):,}")
print(
    f"Unique customers: "
    f"{ml_features['customer_id'].nunique():,}"
)
print(
    f"Prediction dates: "
    f"{ml_features['prediction_date'].nunique():,}"
)

print("\nRFM MISSING VALUES")
print("-" * 50)

for column in [
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average"
]:
    print(
        f"{column}: "
        f"{ml_features[column].isna().sum():,}"
    )

# ------------------------------------------------------------
# 7. RFM SUMMARY
# ------------------------------------------------------------

print("\nRFM SUMMARY")
print("-" * 50)

print(
    ml_features[
        [
            "recency_days",
            "frequency",
            "monetary_total",
            "monetary_average"
        ]
    ].describe().round(2).to_string()
)

# ------------------------------------------------------------
# 8. CHECK LOGICAL VALUES
# ------------------------------------------------------------

negative_recency = (
    ml_features["recency_days"] < 0
).sum()

negative_frequency = (
    ml_features["frequency"] < 0
).sum()

negative_monetary = (
    ml_features["monetary_total"] < 0
).sum()

print("\nRFM LOGICAL VALIDATION")
print("-" * 50)
print(f"Negative recency values: {negative_recency}")
print(f"Negative frequency values: {negative_frequency}")
print(f"Negative monetary values: {negative_monetary}")

# ------------------------------------------------------------
# 9. VALIDATE ZERO-PURCHASE CUSTOMERS
# ------------------------------------------------------------
# Customers with frequency = 0 should have zero monetary
# total and no recency value.

zero_frequency = ml_features["frequency"].eq(0)

zero_frequency_errors = (
    ml_features.loc[
        zero_frequency,
        "monetary_total"
    ] != 0
).sum()

zero_frequency_recency_errors = (
    ml_features.loc[
        zero_frequency,
        "recency_days"
    ].notna()
).sum()

print("\nZERO-PURCHASE VALIDATION")
print("-" * 50)
print(
    "Frequency = 0 but monetary_total != 0: "
    f"{zero_frequency_errors}"
)
print(
    "Frequency = 0 but recency is populated: "
    f"{zero_frequency_recency_errors}"
)

# ------------------------------------------------------------
# 10. CHECK PREDICTION-DATE DUPLICATES
# ------------------------------------------------------------

duplicate_count = (
    ml_features
    .duplicated(
        subset=["customer_id", "prediction_date"]
    )
    .sum()
)

print("\nOBSERVATION UNIQUENESS")
print("-" * 50)
print(
    f"Duplicate customer-prediction observations: "
    f"{duplicate_count}"
)

# ------------------------------------------------------------
# 11. FINAL ASSERTIONS
# ------------------------------------------------------------

assert len(ml_features) == len(ml_target_df)

assert (
    ml_features["customer_id"].nunique()
    == ml_target_df["customer_id"].nunique()
)

assert (
    ml_features["prediction_date"].nunique()
    == len(ML_PREDICTION_DATES)
)

assert negative_recency == 0
assert negative_frequency == 0
assert negative_monetary == 0

assert zero_frequency_errors == 0
assert zero_frequency_recency_errors == 0

assert duplicate_count == 0

print("\n" + "=" * 70)
print("CELL 40 RESULT: HISTORICAL PURCHASE/RFM FEATURES CREATED")
print("=" * 70)

print("✓ Historical Completed orders used only up to each prediction date")
print("✓ Recency calculated from historical purchasing behaviour")
print("✓ Frequency calculated from unique Completed orders")
print("✓ Monetary total and average calculated historically")
print("✓ Zero-purchase customers preserved")
print("✓ No negative RFM values detected")
print("✓ No duplicate customer-prediction observations")
print("✓ Full 175,099-row ML population retained")
print("=" * 70)

CELL 40: HISTORICAL PURCHASE / RFM FEATURES

HISTORICAL RFM DATASET
--------------------------------------------------
Rows: 175,099
Unique customers: 5,564
Prediction dates: 55

RFM MISSING VALUES
--------------------------------------------------
recency_days: 51,559
frequency: 0
monetary_total: 0
monetary_average: 51,559

RFM SUMMARY
--------------------------------------------------
       recency_days  frequency  monetary_total  monetary_average
count     123540.00  175099.00       175099.00         123540.00
mean         224.82       2.07         3173.66           1538.40
std          222.17       1.99         3482.81            841.11
min            0.00       0.00            0.00              8.09
25%           62.00       0.00            0.00            982.13
50%          155.00       2.00         2191.38           1419.67
75%          319.00       3.00         5136.63           1955.03
max         1934.00      14.00        35395.37           8417.84

RFM LOGICAL VALIDATION
-

In [ ]:
# ============================================================
# CELL 41: ADD CUSTOMER ATTRIBUTES TO ML DATASET
# ============================================================

print("=" * 70)
print("CELL 41: CUSTOMER ATTRIBUTES")
print("=" * 70)

# ------------------------------------------------------------
# 1. SELECT CUSTOMER ATTRIBUTES
# ------------------------------------------------------------

customer_attributes = customers[
    [
        "customer_id",
        "membership_tier",
        "preferred_channel",
        "state",
        "referral_source",
        "marketing_opt_in"
    ]
].copy()

# ------------------------------------------------------------
# 2. HANDLE MISSING REFERRAL SOURCE
# ------------------------------------------------------------
# Missing referral source was identified during EDA.
#
# We do not drop these customers.
# Instead, "Unknown" explicitly represents missing referral
# information.

customer_attributes["referral_source"] = (
    customer_attributes["referral_source"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 3. MERGE CUSTOMER ATTRIBUTES
# ------------------------------------------------------------

ml_features = ml_features.merge(
    customer_attributes,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 4. CHECK MERGE COVERAGE
# ------------------------------------------------------------

print("\nMERGE VALIDATION")
print("-" * 50)

print(
    f"ML observations after merge: "
    f"{len(ml_features):,}"
)

print(
    f"Unique customers after merge: "
    f"{ml_features['customer_id'].nunique():,}"
)

missing_customer_attributes = (
    ml_features[
        [
            "membership_tier",
            "preferred_channel",
            "state",
            "referral_source",
            "marketing_opt_in"
        ]
    ]
    .isna()
    .sum()
)

print("\nMISSING CUSTOMER ATTRIBUTES")
print("-" * 50)
print(missing_customer_attributes)

# ------------------------------------------------------------
# 5. DISPLAY CATEGORY DISTRIBUTIONS
# ------------------------------------------------------------

print("\nMEMBERSHIP TIER DISTRIBUTION")
print("-" * 50)
print(
    ml_features["membership_tier"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPREFERRED CHANNEL DISTRIBUTION")
print("-" * 50)
print(
    ml_features["preferred_channel"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nREFERRAL SOURCE DISTRIBUTION")
print("-" * 50)
print(
    ml_features["referral_source"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMARKETING OPT-IN DISTRIBUTION")
print("-" * 50)
print(
    ml_features["marketing_opt_in"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# 6. CHECK EXPECTED CUSTOMER POPULATION
# ------------------------------------------------------------

expected_ml_rows = len(ml_target_df)

assert len(ml_features) == expected_ml_rows

assert (
    ml_features["customer_id"].nunique()
    == ml_target_df["customer_id"].nunique()
)

assert (
    ml_features["prediction_date"].nunique()
    == len(ML_PREDICTION_DATES)
)

# Customer attributes should be populated for every observation
# if the underlying customer record contains the attribute.
#
# Referral source is explicitly filled as Unknown.

assert (
    ml_features["referral_source"].isna().sum()
    == 0
)

# ------------------------------------------------------------
# 7. CHECK FOR DUPLICATES AFTER MERGE
# ------------------------------------------------------------

duplicate_customer_dates = (
    ml_features
    .duplicated(
        subset=["customer_id", "prediction_date"]
    )
    .sum()
)

print("\nDUPLICATE CHECK")
print("-" * 50)
print(
    f"Duplicate customer-prediction observations: "
    f"{duplicate_customer_dates}"
)

assert duplicate_customer_dates == 0

# ------------------------------------------------------------
# 8. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 41 RESULT: CUSTOMER ATTRIBUTES ADDED")
print("=" * 70)

print("✓ Membership tier added")
print("✓ Preferred channel added")
print("✓ State added")
print("✓ Referral source added")
print("✓ Missing referral source preserved as 'Unknown'")
print("✓ Marketing opt-in added")
print("✓ ML population size preserved")
print("✓ No duplicate customer-prediction observations")
print("=" * 70)

CELL 41: CUSTOMER ATTRIBUTES

MERGE VALIDATION
--------------------------------------------------
ML observations after merge: 175,099
Unique customers after merge: 5,564

MISSING CUSTOMER ATTRIBUTES
--------------------------------------------------
membership_tier          0
preferred_channel        0
state                    0
referral_source          0
marketing_opt_in     34895
dtype: int64

MEMBERSHIP TIER DISTRIBUTION
--------------------------------------------------
membership_tier
Bronze      79585
Gold        31776
Platinum    11877
Silver      51861
Name: count, dtype: int64

PREFERRED CHANNEL DISTRIBUTION
--------------------------------------------------
preferred_channel
In-Store      59182
Mobile App    59437
Online        56480
Name: count, dtype: int64

REFERRAL SOURCE DISTRIBUTION
--------------------------------------------------
referral_source
Email Campaign     16985
Friend Referral    17700
Influencer         17088
Search Engine      16975
Social Media       193

In [ ]:
# ============================================================
# CELL 42: CUSTOMER ATTRIBUTE TEMPORAL / DATA-QUALITY AUDIT
# ============================================================

print("=" * 70)
print("CELL 42: CUSTOMER ATTRIBUTE TEMPORAL AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. CUSTOMER-LEVEL ATTRIBUTE COMPLETENESS
# ------------------------------------------------------------
# The customers table contains one record per customer.
# Therefore, we cannot directly observe historical changes
# to customer attributes.

attribute_columns = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in"
]

customer_attribute_audit = pd.DataFrame({
    "attribute": attribute_columns,
    "missing_customers": [
        customers[col].isna().sum()
        for col in attribute_columns
    ],
    "missing_pct": [
        customers[col].isna().mean() * 100
        for col in attribute_columns
    ],
    "unique_values": [
        customers[col].nunique(dropna=True)
        for col in attribute_columns
    ]
})

print("\nCUSTOMER-LEVEL ATTRIBUTE AUDIT")
print("-" * 50)
print(
    customer_attribute_audit
    .round({"missing_pct": 2})
    .to_string(index=False)
)

# ------------------------------------------------------------
# 2. CHECK THAT EACH CUSTOMER HAS ONLY ONE MASTER RECORD
# ------------------------------------------------------------

duplicate_customer_records = (
    customers["customer_id"]
    .duplicated()
    .sum()
)

print("\nCUSTOMER MASTER-RECORD CHECK")
print("-" * 50)
print(
    f"Duplicate customer records: "
    f"{duplicate_customer_records}"
)

# ------------------------------------------------------------
# 3. CLASSIFY FEATURES BY TEMPORAL RISK
# ------------------------------------------------------------
# Because the source contains no historical versions of these
# attributes, we explicitly document our assumption.
#
# Fixed / origin-based:
#   referral_source is tied to acquisition.
#
# Potentially changing:
#   membership_tier
#   preferred_channel
#   state
#   marketing_opt_in
#
# We will not pretend that the source provides historical
# versions when it does not.

attribute_governance = pd.DataFrame({
    "attribute": [
        "membership_tier",
        "preferred_channel",
        "state",
        "referral_source",
        "marketing_opt_in"
    ],
    "temporal_status": [
        "Potentially changing",
        "Potentially changing",
        "Potentially changing",
        "Origin/acquisition attribute",
        "Potentially changing"
    ],
    "historical_versions_available": [
        "No",
        "No",
        "No",
        "No",
        "No"
    ]
})

print("\nATTRIBUTE TEMPORAL GOVERNANCE")
print("-" * 50)
print(
    attribute_governance.to_string(index=False)
)

# ------------------------------------------------------------
# 4. RECORD THE DATA LIMITATION
# ------------------------------------------------------------

CUSTOMER_ATTRIBUTE_LIMITATION = (
    "The customer master table provides one current record per "
    "customer and does not contain historical versions of "
    "potentially changing attributes. Therefore, historical ML "
    "snapshots cannot independently verify the exact historical "
    "state of membership tier, preferred channel, state, or "
    "marketing opt-in at every prediction date."
)

print("\nDATA LIMITATION")
print("-" * 50)
print(CUSTOMER_ATTRIBUTE_LIMITATION)

# ------------------------------------------------------------
# 5. ML GOVERNANCE DECISION
# ------------------------------------------------------------

print("\nML GOVERNANCE DECISION")
print("-" * 50)

print(
    "For the first leakage-controlled model dataset, we will "
    "retain these customer attributes as master-data predictors "
    "but explicitly document the historical-availability "
    "limitation."
)

print(
    "Referral source is treated as an acquisition attribute "
    "because it describes how the customer was acquired."
)

print(
    "Potentially changing attributes will be flagged as a "
    "data limitation and assessed during model interpretation."
)

print(
    "Marketing opt-in missing values will NOT be silently "
    "converted to Yes or No."
)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

assert duplicate_customer_records == 0
assert len(customer_attribute_audit) == 5
assert len(attribute_governance) == 5

print("\n" + "=" * 70)
print("CELL 42 RESULT: CUSTOMER ATTRIBUTE GOVERNANCE DOCUMENTED")
print("=" * 70)

print("✓ Customer master table contains one record per customer")
print("✓ Attribute completeness quantified")
print("✓ Historical-version limitation explicitly documented")
print("✓ Referral source treated as acquisition information")
print("✓ Potentially changing attributes flagged")
print("✓ Marketing opt-in missingness preserved")
print("=" * 70)

CELL 42: CUSTOMER ATTRIBUTE TEMPORAL AUDIT

CUSTOMER-LEVEL ATTRIBUTE AUDIT
--------------------------------------------------
        attribute  missing_customers  missing_pct  unique_values
  membership_tier                  0         0.00              4
preferred_channel                  0         0.00              3
            state                  0         0.00             20
  referral_source               1705        28.00              7
 marketing_opt_in               1191        19.56              2

CUSTOMER MASTER-RECORD CHECK
--------------------------------------------------
Duplicate customer records: 0

ATTRIBUTE TEMPORAL GOVERNANCE
--------------------------------------------------
        attribute              temporal_status historical_versions_available
  membership_tier         Potentially changing                            No
preferred_channel         Potentially changing                            No
            state         Potentially changing              

In [ ]:

# ================================================================
# CELL 43: BUILD LEAKAGE-CONTROLLED SUPPORT FEATURES
# ================================================================
# Purpose:
# Calculate customer support history using only tickets dated on
# or before each historical prediction date.
#
# Leakage prevention:
# A ticket dated after the prediction date must not contribute to
# any feature for that customer-prediction observation.

import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 43: LEAKAGE-CONTROLLED SUPPORT FEATURE ENGINEERING")
print("=" * 70)

# Work with a copy so the original support dataset remains unchanged.
support_for_ml = support_tickets.copy()

# Confirm that the required columns are present.
required_support_columns = [
    "customer_id",
    "ticket_id",
    "ticket_date",
    "satisfaction_score",
    "resolution_time_hours"
]

missing_columns = [
    col for col in required_support_columns
    if col not in support_for_ml.columns
]

assert not missing_columns, (
    f"Missing required support columns: {missing_columns}"
)

# Parse dates and numeric measurements safely.
support_for_ml["ticket_date"] = pd.to_datetime(
    support_for_ml["ticket_date"], errors="coerce"
)

support_for_ml["satisfaction_score"] = pd.to_numeric(
    support_for_ml["satisfaction_score"], errors="coerce"
)

support_for_ml["resolution_time_hours"] = pd.to_numeric(
    support_for_ml["resolution_time_hours"], errors="coerce"
)

# Only dated tickets can be assigned safely to a prediction date.
support_for_ml = support_for_ml.dropna(
    subset=["customer_id", "ticket_id", "ticket_date"]
).copy()

# Use the already-created historical prediction dataset.
# Each row represents one customer at one prediction date.
support_snapshot_keys = ml_features[
    ["customer_id", "prediction_date"]
].drop_duplicates().copy()

support_snapshot_keys["prediction_date"] = pd.to_datetime(
    support_snapshot_keys["prediction_date"]
)

print("\nINPUT CHECK")
print("-" * 50)
print(f"Historical customer-prediction observations: {len(support_snapshot_keys):,}")
print(f"Dated support tickets available: {len(support_for_ml):,}")
print(
    "Earliest dated ticket:",
    support_for_ml["ticket_date"].min().date()
)
print(
    "Latest dated ticket:",
    support_for_ml["ticket_date"].max().date()
)

# Join customer snapshots to that customer's tickets.
# This creates candidate ticket/snapshot pairs for filtering.
snapshot_ticket_pairs = support_snapshot_keys.merge(
    support_for_ml[
        [
            "customer_id",
            "ticket_id",
            "ticket_date",
            "satisfaction_score",
            "resolution_time_hours"
        ]
    ],
    on="customer_id",
    how="left"
)

# Keep only tickets known by the relevant prediction date.
prior_ticket_pairs = snapshot_ticket_pairs.loc[
    snapshot_ticket_pairs["ticket_date"].notna()
    & (
        snapshot_ticket_pairs["ticket_date"]
        <= snapshot_ticket_pairs["prediction_date"]
    )
].copy()

# Aggregate only the permitted historical tickets.
support_features = (
    prior_ticket_pairs
    .groupby(["customer_id", "prediction_date"], as_index=False)
    .agg(
        support_ticket_count=("ticket_id", "nunique"),
        satisfaction_average=("satisfaction_score", "mean"),
        resolution_time_average=("resolution_time_hours", "mean")
    )
)

# Attach features to every customer-prediction observation.
ml_features = ml_features.drop(
    columns=[
        "support_ticket_count",
        "satisfaction_average",
        "resolution_time_average"
    ],
    errors="ignore"
)

ml_features = ml_features.merge(
    support_features,
    on=["customer_id", "prediction_date"],
    how="left",
    validate="one_to_one"
)

# No qualifying tickets means zero tickets.
# Satisfaction and resolution averages remain NaN when unavailable.
ml_features["support_ticket_count"] = (
    ml_features["support_ticket_count"].fillna(0).astype(int)
)

print("\nSUPPORT FEATURE SUMMARY")
print("-" * 50)

print(
    ml_features[
        [
            "support_ticket_count",
            "satisfaction_average",
            "resolution_time_average"
        ]
    ].describe().round(3)
)

print("\nMISSING VALUES")
print("-" * 50)
print(
    ml_features[
        [
            "support_ticket_count",
            "satisfaction_average",
            "resolution_time_average"
        ]
    ].isna().sum()
)

# Validate that the merge has not changed the dataset's row count
# or created duplicate customer-prediction observations.
assert len(ml_features) == len(support_snapshot_keys), (
    "Unexpected change in the number of historical observations."
)

assert not ml_features.duplicated(
    ["customer_id", "prediction_date"]
).any(), "Duplicate customer-prediction observations detected."

assert ml_features["support_ticket_count"].notna().all(), (
    "Support ticket counts should not be missing."
)

assert (ml_features["support_ticket_count"] >= 0).all(), (
    "Negative support ticket counts detected."
)

# Independent leakage check:
# Every ticket contributing to a feature must be dated no later
# than the prediction date for that customer-prediction pair.
if not prior_ticket_pairs.empty:
    assert (
        prior_ticket_pairs["ticket_date"]
        <= prior_ticket_pairs["prediction_date"]
    ).all(), "Future support ticket leakage detected."

print("\nVALIDATION")
print("-" * 50)
print(f"Rows retained: {len(ml_features):,}")
print(
    "Duplicate customer-prediction observations:",
    ml_features.duplicated(
        ["customer_id", "prediction_date"]
    ).sum()
)
print(
    "Future tickets included in feature construction:",
    int(
        (
            prior_ticket_pairs["ticket_date"]
            > prior_ticket_pairs["prediction_date"]
        ).sum()
    )
)

print("\nCELL 43 RESULT: SUPPORT FEATURES CREATED AND VALIDATED")
print("=" * 70)

CELL 43: LEAKAGE-CONTROLLED SUPPORT FEATURE ENGINEERING

INPUT CHECK
--------------------------------------------------
Historical customer-prediction observations: 175,099
Dated support tickets available: 5,200
Earliest dated ticket: 2024-09-03
Latest dated ticket: 2026-06-30

SUPPORT FEATURE SUMMARY
--------------------------------------------------
       support_ticket_count  satisfaction_average  resolution_time_average
count            175099.000             19883.000                21967.000
mean                  0.153                 3.928                   18.170
std                   0.443                 1.102                   18.235
min                   0.000                 1.000                    0.000
25%                   0.000                 3.000                    6.200
50%                   0.000                 4.000                   13.200
75%                   0.000                 5.000                   24.700
max                   5.000                 5.

In [ ]:
# ================================================================
# CELL 44A-2: ATTACH VALIDATED ML TARGET
# ================================================================
# Purpose:
# Attach the already-created and validated `ml_target` from
# Cell 37 to the leakage-controlled ML feature dataset.
#
# We are NOT recreating the target.
# We are simply joining the validated target to the features.

print("=" * 70)
print("CELL 44A-2: ATTACH VALIDATED ML TARGET")
print("=" * 70)

# ------------------------------------------------
# 1. Create the validated target lookup
# ------------------------------------------------

target_lookup = ml_target_df[
    [
        "customer_id",
        "prediction_date",
        "ml_target"
    ]
].copy()

# ------------------------------------------------
# 2. Check target uniqueness
# ------------------------------------------------

duplicate_target_keys = target_lookup.duplicated(
    ["customer_id", "prediction_date"]
).sum()

print("\nTARGET LOOKUP CHECK")
print("-" * 50)

print(
    "Target rows:",
    f"{len(target_lookup):,}"
)

print(
    "Duplicate customer-prediction target keys:",
    duplicate_target_keys
)

assert duplicate_target_keys == 0, (
    "Duplicate target records detected."
)

# ------------------------------------------------
# 3. Remove any previous target column if present
# ------------------------------------------------

ml_features = ml_features.drop(
    columns=["ml_target"],
    errors="ignore"
)

# ------------------------------------------------
# 4. Attach the validated target
# ------------------------------------------------

ml_features = ml_features.merge(
    target_lookup,
    on=["customer_id", "prediction_date"],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------
# 5. Inspect target distribution
# ------------------------------------------------

print("\nTARGET DISTRIBUTION AFTER MERGE")
print("-" * 50)

print(
    ml_features["ml_target"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------
# 6. Validate target completeness
# ------------------------------------------------

missing_target = ml_features["ml_target"].isna().sum()

print("\nTARGET VALIDATION")
print("-" * 50)

print(
    "Missing target:",
    missing_target
)

assert missing_target == 0, (
    "Some feature observations do not have a target."
)

assert set(
    ml_features["ml_target"].unique()
).issubset({0, 1}), (
    "Target contains values other than 0 and 1."
)

# ------------------------------------------------
# 7. Validate dataset integrity
# ------------------------------------------------

print("\nDATASET INTEGRITY")
print("-" * 50)

print(
    "Feature dataset rows:",
    f"{len(ml_features):,}"
)

print(
    "Target lookup rows:",
    f"{len(target_lookup):,}"
)

print(
    "Unique customers:",
    f"{ml_features['customer_id'].nunique():,}"
)

print(
    "Unique prediction dates:",
    f"{ml_features['prediction_date'].nunique():,}"
)

duplicate_feature_keys = ml_features.duplicated(
    ["customer_id", "prediction_date"]
).sum()

print(
    "Duplicate customer-prediction observations:",
    duplicate_feature_keys
)

# ------------------------------------------------
# 8. Final assertions
# ------------------------------------------------

assert len(ml_features) == len(target_lookup), (
    "Feature row count changed unexpectedly after target merge."
)

assert duplicate_feature_keys == 0, (
    "Duplicate customer-prediction observations detected."
)

print("\nCELL 44A-2 RESULT: VALIDATED ML TARGET ATTACHED SUCCESSFULLY")
print("=" * 70)

CELL 44A-2: ATTACH VALIDATED ML TARGET

TARGET LOOKUP CHECK
--------------------------------------------------
Target rows: 175,099
Duplicate customer-prediction target keys: 0

TARGET DISTRIBUTION AFTER MERGE
--------------------------------------------------
ml_target
0     66269
1    108830
Name: count, dtype: int64

TARGET VALIDATION
--------------------------------------------------
Missing target: 0

DATASET INTEGRITY
--------------------------------------------------
Feature dataset rows: 175,099
Target lookup rows: 175,099
Unique customers: 5,564
Unique prediction dates: 55
Duplicate customer-prediction observations: 0

CELL 44A-2 RESULT: VALIDATED ML TARGET ATTACHED SUCCESSFULLY


In [ ]:
# ================================================================
# CELL 44: TIME-AWARE TENURE FEATURE & FINAL FEATURE AUDIT
# ================================================================
# Purpose:
# Add customer tenure as of each prediction date and perform a
# complete audit of the historical ML feature dataset.
#
# Leakage prevention:
# Tenure is calculated as:
#     prediction_date - signup_date
#
# This means the model only receives the tenure information that
# would have been known at the prediction date.

print("=" * 70)
print("CELL 44: TIME-AWARE TENURE FEATURE & FINAL FEATURE AUDIT")
print("=" * 70)

# ------------------------------------------------
# 1. Confirm required columns
# ------------------------------------------------

required_feature_columns = [
    "customer_id",
    "prediction_date",
    "ml_target",
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in",
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average",
    "support_ticket_count",
    "satisfaction_average",
    "resolution_time_average"
]

missing_required_columns = [
    col for col in required_feature_columns
    if col not in ml_features.columns
]

assert not missing_required_columns, (
    f"Missing required ML feature columns: {missing_required_columns}"
)

# ------------------------------------------------
# 2. Merge signup date
# ------------------------------------------------

signup_lookup = customers[
    ["customer_id", "signup_date"]
].copy()

signup_lookup["signup_date"] = pd.to_datetime(
    signup_lookup["signup_date"],
    errors="coerce"
)

assert not signup_lookup["customer_id"].duplicated().any(), (
    "Duplicate customer IDs found in signup lookup."
)

# Remove previous versions if this cell is re-run.
ml_features = ml_features.drop(
    columns=["signup_date", "tenure_days"],
    errors="ignore"
)

ml_features = ml_features.merge(
    signup_lookup,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------
# 3. Calculate tenure at prediction date
# ------------------------------------------------

ml_features["tenure_days"] = (
    ml_features["prediction_date"]
    - ml_features["signup_date"]
).dt.days

# ------------------------------------------------
# 4. Validate tenure
# ------------------------------------------------

print("\nTENURE VALIDATION")
print("-" * 50)

print(
    "Missing signup dates:",
    ml_features["signup_date"].isna().sum()
)

print(
    "Missing tenure values:",
    ml_features["tenure_days"].isna().sum()
)

print(
    "Negative tenure values:",
    (ml_features["tenure_days"] < 0).sum()
)

print(
    "Minimum tenure:",
    ml_features["tenure_days"].min()
)

print(
    "Median tenure:",
    ml_features["tenure_days"].median()
)

print(
    "Maximum tenure:",
    ml_features["tenure_days"].max()
)

assert not (
    ml_features["tenure_days"] < 0
).any(), (
    "Negative tenure detected."
)

assert ml_features["signup_date"].notna().all(), (
    "Unexpected missing signup dates detected."
)

assert ml_features["tenure_days"].notna().all(), (
    "Unexpected missing tenure values detected."
)

# ------------------------------------------------
# 5. Remove raw signup date
# ------------------------------------------------
# The model needs the derived tenure feature, not the raw date.

ml_features = ml_features.drop(
    columns=["signup_date"]
)

# ------------------------------------------------
# 6. Define final feature set
# ------------------------------------------------

final_feature_columns = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in",
    "tenure_days",
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average",
    "support_ticket_count",
    "satisfaction_average",
    "resolution_time_average"
]

target_column = "ml_target"

# ------------------------------------------------
# 7. Final feature completeness audit
# ------------------------------------------------

print("\nFINAL FEATURE COMPLETENESS AUDIT")
print("-" * 50)

feature_missingness = (
    ml_features[final_feature_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

feature_missingness["missing_pct"] = (
    feature_missingness["missing_count"]
    / len(ml_features)
    * 100
)

print(
    feature_missingness
    .sort_values("missing_count", ascending=False)
    .round(2)
)

# ------------------------------------------------
# 8. Target audit
# ------------------------------------------------

print("\nTARGET AUDIT")
print("-" * 50)

print(
    ml_features[target_column]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nTarget missing:",
    ml_features[target_column].isna().sum()
)

assert ml_features[target_column].notna().all(), (
    "Missing target values detected."
)

assert set(
    ml_features[target_column].unique()
).issubset({0, 1}), (
    "Target contains values other than 0 and 1."
)

# ------------------------------------------------
# 9. Dataset-level integrity checks
# ------------------------------------------------

print("\nDATASET INTEGRITY")
print("-" * 50)

print(
    "Rows:",
    f"{len(ml_features):,}"
)

print(
    "Unique customers:",
    f"{ml_features['customer_id'].nunique():,}"
)

print(
    "Unique prediction dates:",
    f"{ml_features['prediction_date'].nunique():,}"
)

duplicate_feature_keys = ml_features.duplicated(
    ["customer_id", "prediction_date"]
).sum()

print(
    "Duplicate customer-prediction observations:",
    duplicate_feature_keys
)

assert not ml_features.duplicated(
    ["customer_id", "prediction_date"]
).any(), (
    "Duplicate customer-prediction observations detected."
)

assert ml_features["prediction_date"].notna().all(), (
    "Missing prediction dates detected."
)

# ------------------------------------------------
# 10. Final feature list
# ------------------------------------------------

print("\nFINAL ML FEATURE SET")
print("-" * 50)

for i, feature in enumerate(final_feature_columns, start=1):
    print(f"{i:2}. {feature}")

print(f"\nTarget: {target_column}")

print("\nCELL 44 RESULT: TIME-AWARE FEATURE AUDIT PASSED")
print("=" * 70)

CELL 44: TIME-AWARE TENURE FEATURE & FINAL FEATURE AUDIT

TENURE VALIDATION
--------------------------------------------------
Missing signup dates: 0
Missing tenure values: 0
Negative tenure values: 0
Minimum tenure: 0
Median tenure: 584.0
Maximum tenure: 1946

FINAL FEATURE COMPLETENESS AUDIT
--------------------------------------------------
                         missing_count  missing_pct
satisfaction_average            155216        88.64
resolution_time_average         153132        87.45
recency_days                     51559        29.45
monetary_average                 51559        29.45
marketing_opt_in                 34895        19.93
referral_source                      0         0.00
state                                0         0.00
preferred_channel                    0         0.00
membership_tier                      0         0.00
monetary_total                       0         0.00
frequency                            0         0.00
tenure_days                  

In [ ]:
# ================================================================
# CELL 45: MISSING-VALUE STRATEGY & FEATURE-TYPE CLASSIFICATION
# ================================================================
# Purpose:
# 1. Classify the ML features into categorical and numerical groups.
# 2. Quantify missingness in each feature.
# 3. Document the reason for missing values.
# 4. Define the planned treatment for each feature.
#
# Important:
# NO IMPUTATION IS PERFORMED IN THIS CELL.
#
# Actual imputation will later be performed inside a scikit-learn
# preprocessing pipeline fitted only on the training data.
# This prevents data leakage.

print("=" * 70)
print("CELL 45: MISSING-VALUE STRATEGY & FEATURE-TYPE CLASSIFICATION")
print("=" * 70)

# ------------------------------------------------
# 1. Define the final feature groups
# ------------------------------------------------

categorical_features = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in"
]

numerical_features = [
    "tenure_days",
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average",
    "support_ticket_count",
    "satisfaction_average",
    "resolution_time_average"
]

target_column = "ml_target"

# ------------------------------------------------
# 2. Confirm feature definitions
# ------------------------------------------------

all_feature_columns = (
    categorical_features +
    numerical_features
)

assert set(all_feature_columns) == set(final_feature_columns), (
    "Feature classification does not match final feature set."
)

assert len(all_feature_columns) == len(
    set(all_feature_columns)
), (
    "Duplicate feature names detected."
)

assert target_column in ml_features.columns, (
    f"Target column '{target_column}' not found."
)

# ------------------------------------------------
# 3. Build missingness audit
# ------------------------------------------------

missingness_audit = (
    ml_features[all_feature_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

missingness_audit["missing_pct"] = (
    missingness_audit["missing_count"]
    / len(ml_features)
    * 100
)

missingness_audit["feature_type"] = [
    "categorical"
    if feature in categorical_features
    else "numerical"
    for feature in missingness_audit.index
]

print("\nFEATURE CLASSIFICATION & MISSINGNESS")
print("-" * 70)

print(
    missingness_audit
    .sort_values("missing_count", ascending=False)
    .round(2)
)

# ------------------------------------------------
# 4. Document the meaning of missing values
# ------------------------------------------------
# These are business/data meanings, not arbitrary replacements.

missing_value_strategy = pd.DataFrame({
    "feature": [
        "membership_tier",
        "preferred_channel",
        "state",
        "referral_source",
        "marketing_opt_in",
        "tenure_days",
        "recency_days",
        "frequency",
        "monetary_total",
        "monetary_average",
        "support_ticket_count",
        "satisfaction_average",
        "resolution_time_average"
    ],

    "feature_type": [
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "numerical",
        "numerical",
        "numerical",
        "numerical",
        "numerical",
        "numerical",
        "numerical",
        "numerical"
    ],

    "missing_meaning": [
        "No missing values in the cleaned customer master.",
        "No missing values in the cleaned customer master.",
        "No missing values after state standardization.",
        "Original referral information was unavailable.",
        "Marketing preference was not recorded.",
        "No missing signup dates; therefore tenure is available.",
        "No completed-order history available by the prediction date.",
        "No completed orders available by the prediction date.",
        "No completed orders available by the prediction date.",
        "No completed-order history available by the prediction date.",
        "No support tickets recorded by the prediction date.",
        "No recorded satisfaction score available.",
        "No recorded resolution-time observation available."
    ],

    "planned_treatment": [
        "No imputation required.",
        "No imputation required.",
        "No imputation required.",
        "Retain explicit 'Unknown' category.",
        "Categorical imputation with an explicit 'Unknown' category.",
        "No imputation required.",
        "Median imputation inside training-only pipeline + missingness indicator.",
        "No imputation required; zero represents no completed orders.",
        "No imputation required; zero represents no completed orders.",
        "Median imputation inside training-only pipeline + missingness indicator.",
        "Retain zero because zero means no recorded tickets.",
        "Median imputation inside training-only pipeline + missingness indicator.",
        "Median imputation inside training-only pipeline + missingness indicator."
    ]
})

print("\nMISSING-VALUE GOVERNANCE")
print("-" * 70)

print(missing_value_strategy.to_string(index=False))

# ------------------------------------------------
# 5. Verify planned strategies against actual data
# ------------------------------------------------

print("\nSTRATEGY VALIDATION")
print("-" * 70)

# Referral source should already contain explicit Unknown values
# rather than NaN after Cell 41.
print(
    "Referral source missing values:",
    ml_features["referral_source"].isna().sum()
)

print(
    "Referral source Unknown values:",
    (
        ml_features["referral_source"]
        .astype(str)
        .str.strip()
        .eq("Unknown")
        .sum()
    )
)

# Support ticket count should use zero for customers with
# no recorded support history.
print(
    "Support ticket count zeros:",
    (ml_features["support_ticket_count"] == 0).sum()
)

print(
    "Support ticket count missing:",
    ml_features["support_ticket_count"].isna().sum()
)

# Frequency and monetary total use zero when there is no
# completed-order history.
print(
    "Frequency zeros:",
    (ml_features["frequency"] == 0).sum()
)

print(
    "Monetary total zeros:",
    (ml_features["monetary_total"] == 0).sum()
)

# ------------------------------------------------
# 6. Confirm target is excluded from feature groups
# ------------------------------------------------

assert target_column not in all_feature_columns, (
    "Target has accidentally been included as a predictor."
)

# ------------------------------------------------
# 7. Confirm identifiers are not predictors
# ------------------------------------------------

assert "customer_id" not in all_feature_columns, (
    "customer_id must not be used as a predictive feature."
)

assert "prediction_date" not in all_feature_columns, (
    "prediction_date is not being used directly as a predictor."
)

# ------------------------------------------------
# 8. Final governance summary
# ------------------------------------------------

print("\nFINAL ML PREPROCESSING PLAN")
print("-" * 70)

print(
    "Categorical features:",
    len(categorical_features)
)

for feature in categorical_features:
    print(f"  - {feature}")

print(
    "\nNumerical features:",
    len(numerical_features)
)

for feature in numerical_features:
    print(f"  - {feature}")

print("\nKey principles:")
print("  ✓ No blanket zero-imputation.")
print("  ✓ Meaningful structural zeros are retained.")
print("  ✓ Missing categorical values use explicit Unknown treatment.")
print("  ✓ Numerical missing values will be handled inside the ML pipeline.")
print("  ✓ Missingness indicators will preserve useful information where appropriate.")
print("  ✓ Imputation will be fitted on training data only.")
print("  ✓ Target and identifiers are excluded from predictors.")

print("\nCELL 45 RESULT: MISSING-VALUE STRATEGY DOCUMENTED")
print("=" * 70)

CELL 45: MISSING-VALUE STRATEGY & FEATURE-TYPE CLASSIFICATION

FEATURE CLASSIFICATION & MISSINGNESS
----------------------------------------------------------------------
                         missing_count  missing_pct feature_type
satisfaction_average            155216        88.64    numerical
resolution_time_average         153132        87.45    numerical
recency_days                     51559        29.45    numerical
monetary_average                 51559        29.45    numerical
marketing_opt_in                 34895        19.93  categorical
referral_source                      0         0.00  categorical
state                                0         0.00  categorical
preferred_channel                    0         0.00  categorical
membership_tier                      0         0.00  categorical
monetary_total                       0         0.00    numerical
frequency                            0         0.00    numerical
tenure_days                          0         0.

In [ ]:
# ================================================================
# CELL 46: FORMAL ML FEATURE LEAKAGE AUDIT
# ================================================================
# Purpose:
# Perform a formal audit to ensure that no target, future outcome,
# identifier, or future-derived information has entered the ML
# predictor set.
#
# This is a governance checkpoint before train/test splitting.

print("=" * 70)
print("CELL 46: FORMAL ML FEATURE LEAKAGE AUDIT")
print("=" * 70)

# ------------------------------------------------
# 1. Define the intended predictors
# ------------------------------------------------

predictor_columns = (
    categorical_features +
    numerical_features
)

print("\nINTENDED PREDICTORS")
print("-" * 50)

for i, feature in enumerate(predictor_columns, start=1):
    print(f"{i:2}. {feature}")

# ------------------------------------------------
# 2. Define columns that must NEVER be predictors
# ------------------------------------------------
# These include:
# - target
# - customer identifier
# - prediction date
# - official EDA churn flag
# - future outcome information

forbidden_columns = [
    "ml_target",
    "customer_id",
    "prediction_date",
    "churn_flag",
    "future_completed_orders",
    "future_order_count",
    "future_order_amount",
    "future_revenue",
    "future_channel",
    "future_support_tickets"
]

# ------------------------------------------------
# 3. Check predictor set against forbidden columns
# ------------------------------------------------

forbidden_in_predictors = [
    col for col in forbidden_columns
    if col in predictor_columns
]

print("\nFORBIDDEN-COLUMN CHECK")
print("-" * 50)

if forbidden_in_predictors:
    print("FORBIDDEN COLUMNS FOUND:")
    for col in forbidden_in_predictors:
        print(f"  ✗ {col}")
else:
    print("No forbidden columns found among predictors.")

assert not forbidden_in_predictors, (
    f"Potential leakage columns detected: {forbidden_in_predictors}"
)

# ------------------------------------------------
# 4. Check predictor columns actually exist
# ------------------------------------------------

missing_predictors = [
    col for col in predictor_columns
    if col not in ml_features.columns
]

print("\nPREDICTOR AVAILABILITY CHECK")
print("-" * 50)

if missing_predictors:
    print("Missing predictor columns:")
    for col in missing_predictors:
        print(f"  ✗ {col}")
else:
    print(
        f"All {len(predictor_columns)} predictors are present."
    )

assert not missing_predictors, (
    f"Missing predictor columns: {missing_predictors}"
)

# ------------------------------------------------
# 5. Check target is separate from predictors
# ------------------------------------------------

print("\nTARGET SEPARATION CHECK")
print("-" * 50)

print(
    "Target column:",
    target_column
)

print(
    "Target included among predictors:",
    target_column in predictor_columns
)

assert target_column not in predictor_columns, (
    "Target has been included among predictors."
)

# ------------------------------------------------
# 6. Check customer ID and prediction date
# ------------------------------------------------

print("\nIDENTIFIER / DATE CHECK")
print("-" * 50)

print(
    "customer_id included as predictor:",
    "customer_id" in predictor_columns
)

print(
    "prediction_date included as predictor:",
    "prediction_date" in predictor_columns
)

assert "customer_id" not in predictor_columns
assert "prediction_date" not in predictor_columns

# ------------------------------------------------
# 7. Verify prediction-date range
# ------------------------------------------------

print("\nPREDICTION-DATE RANGE")
print("-" * 50)

print(
    "Earliest prediction date:",
    ml_features["prediction_date"].min().date()
)

print(
    "Latest prediction date:",
    ml_features["prediction_date"].max().date()
)

print(
    "Number of prediction dates:",
    ml_features["prediction_date"].nunique()
)

assert (
    ml_features["prediction_date"].min()
    >= pd.Timestamp("2021-07-01")
)

assert (
    ml_features["prediction_date"].max()
    <= pd.Timestamp("2026-01-01")
)

# ------------------------------------------------
# 8. Validate tenure cannot use future signup information
# ------------------------------------------------

print("\nTENURE LEAKAGE CHECK")
print("-" * 50)

# Reconstruct signup date temporarily for validation only.
signup_check = customers[
    ["customer_id", "signup_date"]
].copy()

signup_check["signup_date"] = pd.to_datetime(
    signup_check["signup_date"],
    errors="coerce"
)

tenure_check = ml_features[
    ["customer_id", "prediction_date", "tenure_days"]
].merge(
    signup_check,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

recalculated_tenure = (
    tenure_check["prediction_date"]
    - tenure_check["signup_date"]
).dt.days

tenure_mismatches = (
    recalculated_tenure
    != tenure_check["tenure_days"]
).sum()

print(
    "Tenure calculation mismatches:",
    tenure_mismatches
)

assert tenure_mismatches == 0, (
    "Tenure feature does not match the prediction-date calculation."
)

assert (
    tenure_check["signup_date"]
    <= tenure_check["prediction_date"]
).all(), (
    "Signup date occurs after prediction date."
)

# ------------------------------------------------
# 9. Final predictor audit
# ------------------------------------------------

print("\nFINAL PREDICTOR AUDIT")
print("-" * 50)

print(
    "Total predictors:",
    len(predictor_columns)
)

print(
    "Categorical predictors:",
    len(categorical_features)
)

print(
    "Numerical predictors:",
    len(numerical_features)
)

print(
    "Target:",
    target_column
)

print(
    "Customer identifier excluded:",
    "customer_id" not in predictor_columns
)

print(
    "Prediction date excluded:",
    "prediction_date" not in predictor_columns
)

print(
    "Future outcome fields excluded:",
    not any(
        col in predictor_columns
        for col in forbidden_columns
        if col != target_column
    )
)

# ------------------------------------------------
# 10. Final result
# ------------------------------------------------

print("\nCELL 46 RESULT: FORMAL LEAKAGE AUDIT PASSED")
print("=" * 70)

CELL 46: FORMAL ML FEATURE LEAKAGE AUDIT

INTENDED PREDICTORS
--------------------------------------------------
 1. membership_tier
 2. preferred_channel
 3. state
 4. referral_source
 5. marketing_opt_in
 6. tenure_days
 7. recency_days
 8. frequency
 9. monetary_total
10. monetary_average
11. support_ticket_count
12. satisfaction_average
13. resolution_time_average

FORBIDDEN-COLUMN CHECK
--------------------------------------------------
No forbidden columns found among predictors.

PREDICTOR AVAILABILITY CHECK
--------------------------------------------------
All 13 predictors are present.

TARGET SEPARATION CHECK
--------------------------------------------------
Target column: ml_target
Target included among predictors: False

IDENTIFIER / DATE CHECK
--------------------------------------------------
customer_id included as predictor: False
prediction_date included as predictor: False

PREDICTION-DATE RANGE
--------------------------------------------------
Earliest prediction 

In [ ]:
# ================================================================
# CELL 47: TIME-BASED TRAIN / VALIDATION / TEST SPLIT
# ================================================================
#
# Purpose:
# Split historical ML observations chronologically rather than
# randomly.
#
# Why?
# The model is intended to predict future customer behaviour.
# A chronological split better simulates real-world deployment
# and reduces the risk of evaluating the model on information
# patterns from the same time period used for training.
#
# We will NOT use prediction_date as a model predictor.
# It is used only to create the temporal split.
# ================================================================

print("=" * 70)
print("CELL 47: TIME-BASED TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)

# ------------------------------------------------
# 1. Confirm prediction dates
# ------------------------------------------------

prediction_dates = sorted(
    ml_features["prediction_date"].dropna().unique()
)

print("\nAVAILABLE PREDICTION DATES")
print("-" * 50)
print(f"Number of prediction dates: {len(prediction_dates)}")
print(f"Earliest date: {prediction_dates[0]}")
print(f"Latest date:   {prediction_dates[-1]}")

# ------------------------------------------------
# 2. Define chronological split points
# ------------------------------------------------
#
# Training:
# July 2021 – December 2024
#
# Validation:
# January 2025 – June 2025
#
# Test:
# July 2025 – January 2026
#
# The test period is kept completely untouched
# until final model evaluation.
# ------------------------------------------------

TRAIN_END = pd.Timestamp("2024-12-01")
VALIDATION_END = pd.Timestamp("2025-06-01")

train_mask = ml_features["prediction_date"] <= TRAIN_END

validation_mask = (
    (ml_features["prediction_date"] > TRAIN_END)
    & (ml_features["prediction_date"] <= VALIDATION_END)
)

test_mask = ml_features["prediction_date"] > VALIDATION_END

train_df = ml_features.loc[train_mask].copy()
validation_df = ml_features.loc[validation_mask].copy()
test_df = ml_features.loc[test_mask].copy()

# ------------------------------------------------
# 3. Display split information
# ------------------------------------------------

print("\nTIME-BASED SPLIT")
print("-" * 50)

print(
    f"Training period:   "
    f"{train_df['prediction_date'].min().date()} "
    f"to {train_df['prediction_date'].max().date()}"
)

print(
    f"Validation period: "
    f"{validation_df['prediction_date'].min().date()} "
    f"to {validation_df['prediction_date'].max().date()}"
)

print(
    f"Test period:       "
    f"{test_df['prediction_date'].min().date()} "
    f"to {test_df['prediction_date'].max().date()}"
)

print("\nOBSERVATION COUNTS")
print("-" * 50)
print(f"Training observations:   {len(train_df):,}")
print(f"Validation observations: {len(validation_df):,}")
print(f"Test observations:       {len(test_df):,}")
print(f"Total observations:      {len(train_df) + len(validation_df) + len(test_df):,}")

# ------------------------------------------------
# 4. Prediction-date counts
# ------------------------------------------------

print("\nPREDICTION-DATE COUNTS")
print("-" * 50)

print(
    f"Training dates:   "
    f"{train_df['prediction_date'].nunique()}"
)

print(
    f"Validation dates: "
    f"{validation_df['prediction_date'].nunique()}"
)

print(
    f"Test dates:       "
    f"{test_df['prediction_date'].nunique()}"
)

# ------------------------------------------------
# 5. Target distribution in each split
# ------------------------------------------------
#
# This is important because churn prevalence changes
# over time. We want to see that distribution explicitly.
# ------------------------------------------------

print("\nTARGET DISTRIBUTION BY SPLIT")
print("-" * 50)

for split_name, split_df in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:

    target_rate = split_df["ml_target"].mean() * 100

    print(
        f"{split_name:<12} "
        f"n={len(split_df):>7,} | "
        f"Churn={target_rate:>6.2f}% | "
        f"Active={100 - target_rate:>6.2f}%"
    )

# ------------------------------------------------
# 6. Check chronological ordering
# ------------------------------------------------

chronology_check = (
    train_df["prediction_date"].max()
    < validation_df["prediction_date"].min()
    and
    validation_df["prediction_date"].max()
    < test_df["prediction_date"].min()
)

print("\nCHRONOLOGICAL ORDER CHECK")
print("-" * 50)
print(f"Training < Validation < Test: {chronology_check}")

# ------------------------------------------------
# 7. Check for missing prediction dates
# ------------------------------------------------

missing_prediction_dates = (
    ml_features["prediction_date"].isna().sum()
)

print("\nPREDICTION-DATE QUALITY CHECK")
print("-" * 50)
print(
    f"Missing prediction dates: "
    f"{missing_prediction_dates}"
)

# ------------------------------------------------
# 8. Check that every observation belongs to
#    exactly one split
# ------------------------------------------------

split_membership = (
    train_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
)

overlap_count = (split_membership > 1).sum()
unassigned_count = (split_membership == 0).sum()

print("\nSPLIT MEMBERSHIP CHECK")
print("-" * 50)
print(f"Overlapping observations: {overlap_count}")
print(f"Unassigned observations:   {unassigned_count}")

# ------------------------------------------------
# 9. Final validation
# ------------------------------------------------

split_pass = (
    len(train_df) > 0
    and len(validation_df) > 0
    and len(test_df) > 0
    and chronology_check
    and missing_prediction_dates == 0
    and overlap_count == 0
    and unassigned_count == 0
    and (
        len(train_df)
        + len(validation_df)
        + len(test_df)
        == len(ml_features)
    )
)

print("\n" + "=" * 70)

if split_pass:
    print("CELL 47 RESULT: TIME-BASED SPLIT PASSED")
else:
    print("CELL 47 RESULT: REVIEW SPLIT BEFORE PROCEEDING")

print("=" * 70)

CELL 47: TIME-BASED TRAIN / VALIDATION / TEST SPLIT

AVAILABLE PREDICTION DATES
--------------------------------------------------
Number of prediction dates: 55
Earliest date: 2021-07-01 00:00:00
Latest date:   2026-01-01 00:00:00

TIME-BASED SPLIT
--------------------------------------------------
Training period:   2021-07-01 to 2024-12-01
Validation period: 2025-01-01 to 2025-06-01
Test period:       2025-07-01 to 2026-01-01

OBSERVATION COUNTS
--------------------------------------------------
Training observations:   109,913
Validation observations: 28,190
Test observations:       36,996
Total observations:      175,099

PREDICTION-DATE COUNTS
--------------------------------------------------
Training dates:   42
Validation dates: 6
Test dates:       7

TARGET DISTRIBUTION BY SPLIT
--------------------------------------------------
Training     n=109,913 | Churn= 60.84% | Active= 39.16%
Validation   n= 28,190 | Churn= 60.13% | Active= 39.87%
Test         n= 36,996 | Churn= 67.60

In [ ]:
# ================================================================
# CELL 48: PREPARE X AND y FOR MACHINE LEARNING
# ================================================================
#
# Purpose:
# Create the predictor matrix (X) and target vector (y) for the
# training, validation and test datasets.
#
# IMPORTANT:
# prediction_date and customer_id are NOT predictors.
# They are retained in the original dataframes for auditing,
# grouping and later interpretation, but excluded from X.
# ================================================================

print("=" * 70)
print("CELL 48: PREPARE X AND y FOR MACHINE LEARNING")
print("=" * 70)

# ------------------------------------------------
# 1. Define the final predictor list
# ------------------------------------------------

predictor_columns = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in",
    "tenure_days",
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average",
    "support_ticket_count",
    "satisfaction_average",
    "resolution_time_average"
]

target_column = "ml_target"

print("\nPREDICTOR SET")
print("-" * 50)

for i, column in enumerate(predictor_columns, start=1):
    print(f"{i:2d}. {column}")

print(f"\nNumber of predictors: {len(predictor_columns)}")
print(f"Target: {target_column}")

# ------------------------------------------------
# 2. Create X and y for each split
# ------------------------------------------------

X_train = train_df[predictor_columns].copy()
y_train = train_df[target_column].copy()

X_validation = validation_df[predictor_columns].copy()
y_validation = validation_df[target_column].copy()

X_test = test_df[predictor_columns].copy()
y_test = test_df[target_column].copy()

# ------------------------------------------------
# 3. Check shapes
# ------------------------------------------------

print("\nDATASET SHAPES")
print("-" * 50)

print(f"X_train:        {X_train.shape}")
print(f"y_train:        {y_train.shape}")

print(f"X_validation:   {X_validation.shape}")
print(f"y_validation:   {y_validation.shape}")

print(f"X_test:         {X_test.shape}")
print(f"y_test:         {y_test.shape}")

# ------------------------------------------------
# 4. Check target values
# ------------------------------------------------

print("\nTARGET VALUES")
print("-" * 50)

print("Training:")
print(y_train.value_counts().sort_index())

print("\nValidation:")
print(y_validation.value_counts().sort_index())

print("\nTest:")
print(y_test.value_counts().sort_index())

# ------------------------------------------------
# 5. Confirm target contains only 0 and 1
# ------------------------------------------------

valid_target_values = {0, 1}

target_values_found = set(
    pd.concat([y_train, y_validation, y_test]).dropna().unique()
)

invalid_target_values = (
    target_values_found - valid_target_values
)

print("\nTARGET VALIDATION")
print("-" * 50)
print(f"Target values found: {sorted(target_values_found)}")
print(f"Invalid target values: {invalid_target_values}")

# ------------------------------------------------
# 6. Confirm there are no missing target values
# ------------------------------------------------

missing_targets = {
    "train": y_train.isna().sum(),
    "validation": y_validation.isna().sum(),
    "test": y_test.isna().sum()
}

print("\nMISSING TARGET CHECK")
print("-" * 50)

for split_name, count in missing_targets.items():
    print(f"{split_name.capitalize():<12}: {count}")

# ------------------------------------------------
# 7. Confirm forbidden columns are excluded
# ------------------------------------------------

forbidden_predictor_columns = {
    "customer_id",
    "prediction_date",
    "ml_target",
    "churn_flag",
    "future_completed_orders",
    "future_order_count",
    "future_order_amount",
    "future_revenue",
    "future_channel",
    "future_support_tickets"
}

forbidden_found = (
    forbidden_predictor_columns
    .intersection(X_train.columns)
)

print("\nFORBIDDEN-COLUMN CHECK")
print("-" * 50)
print(f"Forbidden columns found in X_train: {forbidden_found}")

# ------------------------------------------------
# 8. Confirm train/validation/test row alignment
# ------------------------------------------------

alignment_check = (
    len(X_train) == len(y_train)
    and
    len(X_validation) == len(y_validation)
    and
    len(X_test) == len(y_test)
)

print("\nROW ALIGNMENT CHECK")
print("-" * 50)
print(f"X and y row counts aligned: {alignment_check}")

# ------------------------------------------------
# 9. Check predictor data types
# ------------------------------------------------

categorical_columns = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in"
]

numerical_columns = [
    "tenure_days",
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average",
    "support_ticket_count",
    "satisfaction_average",
    "resolution_time_average"
]

print("\nPREDICTOR DATA TYPES")
print("-" * 50)

print("Categorical:")
for column in categorical_columns:
    print(f"  {column:<25} {X_train[column].dtype}")

print("\nNumerical:")
for column in numerical_columns:
    print(f"  {column:<25} {X_train[column].dtype}")

# ------------------------------------------------
# 10. Final validation
# ------------------------------------------------

cell_48_pass = (
    len(predictor_columns) == 13
    and
    all(column in X_train.columns for column in predictor_columns)
    and
    all(column in X_validation.columns for column in predictor_columns)
    and
    all(column in X_test.columns for column in predictor_columns)
    and
    invalid_target_values == set()
    and
    all(count == 0 for count in missing_targets.values())
    and
    forbidden_found == set()
    and
    alignment_check
    and
    set(categorical_columns).issubset(X_train.columns)
    and
    set(numerical_columns).issubset(X_train.columns)
)

print("\n" + "=" * 70)

if cell_48_pass:
    print("CELL 48 RESULT: X/y PREPARATION PASSED")
else:
    print("CELL 48 RESULT: REVIEW X/y PREPARATION")

print("=" * 70)

CELL 48: PREPARE X AND y FOR MACHINE LEARNING

PREDICTOR SET
--------------------------------------------------
 1. membership_tier
 2. preferred_channel
 3. state
 4. referral_source
 5. marketing_opt_in
 6. tenure_days
 7. recency_days
 8. frequency
 9. monetary_total
10. monetary_average
11. support_ticket_count
12. satisfaction_average
13. resolution_time_average

Number of predictors: 13
Target: ml_target

DATASET SHAPES
--------------------------------------------------
X_train:        (109913, 13)
y_train:        (109913,)
X_validation:   (28190, 13)
y_validation:   (28190,)
X_test:         (36996, 13)
y_test:         (36996,)

TARGET VALUES
--------------------------------------------------
Training:
ml_target
0    43044
1    66869
Name: count, dtype: int64

Validation:
ml_target
0    11239
1    16951
Name: count, dtype: int64

Test:
ml_target
0    11986
1    25010
Name: count, dtype: int64

TARGET VALIDATION
--------------------------------------------------
Target values foun

In [ ]:
# ============================================================
# VALIDATION DATA ALIASES
# ============================================================
# The main train/validation/test preparation uses the names
# X_validation and y_validation. Later model-evaluation cells
# use the shorter names X_val and y_val.
#
# These aliases keep the workflow consistent and reproducible.
# No data is copied from outside the validation period.
# ============================================================

X_val = X_validation.copy()
y_val = y_validation.copy()

print("Validation aliases created successfully")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"Validation target mean: {y_val.mean():.4f}")


In [ ]:
# ================================================================
# CELL 49: BUILD LEAKAGE-SAFE PREPROCESSING PIPELINE
# ================================================================
#
# Purpose:
# Convert the raw ML predictors into a machine-learning-ready
# representation without allowing validation/test information
# to influence preprocessing.
#
# IMPORTANT:
# The preprocessing objects will eventually be fitted ONLY on
# X_train.
#
# This prevents data leakage from:
# - median values calculated from validation/test data
# - category information from validation/test data
# - scaling parameters calculated from validation/test data
#
# The pipeline also preserves meaningful missingness information.
# ================================================================

print("=" * 70)
print("CELL 49: BUILD LEAKAGE-SAFE PREPROCESSING PIPELINE")
print("=" * 70)

# ------------------------------------------------
# 1. Import preprocessing tools
# ------------------------------------------------

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("\nPREPROCESSING COMPONENTS")
print("-" * 50)
print("ColumnTransformer")
print("SimpleImputer")
print("OneHotEncoder")
print("StandardScaler")
print("Pipeline")

# ------------------------------------------------
# 2. Define categorical and numerical columns
# ------------------------------------------------

categorical_features = [
    "membership_tier",
    "preferred_channel",
    "state",
    "referral_source",
    "marketing_opt_in"
]

numerical_features = [
    "tenure_days",
    "recency_days",
    "frequency",
    "monetary_total",
    "monetary_average",
    "support_ticket_count",
    "satisfaction_average",
    "resolution_time_average"
]

print("\nFEATURE GROUPS")
print("-" * 50)
print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features:   {len(numerical_features)}")

# ------------------------------------------------
# 3. Numerical preprocessing
# ------------------------------------------------
#
# Median imputation is calculated from the training data only
# once the pipeline is fitted.
#
# add_indicator=True creates an additional binary indicator
# for numerical columns that contain missing values.
#
# This is useful because:
# "missing satisfaction score" may itself contain information
# rather than simply meaning "average satisfaction".
# ------------------------------------------------

numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# ------------------------------------------------
# 4. Categorical preprocessing
# ------------------------------------------------
#
# Missing categorical values are explicitly represented as
# "Unknown" rather than silently converting them to the
# most common category.
#
# handle_unknown="ignore" prevents the pipeline from failing
# if a category appears in validation/test that was not present
# in training.
# ------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

# ------------------------------------------------
# 5. Combine numerical and categorical processing
# ------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("\nCOLUMN TRANSFORMER")
print("-" * 50)
print("Numerical transformation: median imputation + missing indicators + scaling")
print("Categorical transformation: Unknown imputation + one-hot encoding")
print("Unspecified columns: dropped")

# ------------------------------------------------
# 6. Verify that all predictors are assigned
# ------------------------------------------------

all_preprocessing_features = (
    categorical_features + numerical_features
)

missing_from_preprocessor = set(predictor_columns) - set(
    all_preprocessing_features
)

unexpected_preprocessor_features = set(
    all_preprocessing_features
) - set(predictor_columns)

print("\nFEATURE COVERAGE CHECK")
print("-" * 50)
print(
    f"Predictors not assigned to preprocessing: "
    f"{missing_from_preprocessor}"
)

print(
    f"Unexpected preprocessing features: "
    f"{unexpected_preprocessor_features}"
)

# ------------------------------------------------
# 7. Check that the preprocessor has NOT been fitted yet
# ------------------------------------------------
#
# We deliberately do not call:
#
#     preprocessor.fit(X_train)
#
# yet.
#
# The next modelling step will fit preprocessing inside the
# model pipeline using training data only.
# ------------------------------------------------

is_unfitted = not hasattr(
    preprocessor,
    "transformers_"
)

print("\nFIT STATUS CHECK")
print("-" * 50)
print(
    f"Preprocessor currently unfitted: "
    f"{is_unfitted}"
)

# ------------------------------------------------
# 8. Final validation
# ------------------------------------------------

cell_49_pass = (
    len(categorical_features) == 5
    and len(numerical_features) == 8
    and len(all_preprocessing_features) == 13
    and missing_from_preprocessor == set()
    and unexpected_preprocessor_features == set()
    and is_unfitted
)

print("\n" + "=" * 70)

if cell_49_pass:
    print("CELL 49 RESULT: PREPROCESSING PIPELINE DEFINITION PASSED")
else:
    print("CELL 49 RESULT: REVIEW PREPROCESSING DEFINITION")

print("=" * 70)

CELL 49: BUILD LEAKAGE-SAFE PREPROCESSING PIPELINE

PREPROCESSING COMPONENTS
--------------------------------------------------
ColumnTransformer
SimpleImputer
OneHotEncoder
StandardScaler
Pipeline

FEATURE GROUPS
--------------------------------------------------
Categorical features: 5
Numerical features:   8

COLUMN TRANSFORMER
--------------------------------------------------
Numerical transformation: median imputation + missing indicators + scaling
Categorical transformation: Unknown imputation + one-hot encoding
Unspecified columns: dropped

FEATURE COVERAGE CHECK
--------------------------------------------------
Predictors not assigned to preprocessing: set()
Unexpected preprocessing features: set()

FIT STATUS CHECK
--------------------------------------------------
Preprocessor currently unfitted: True

CELL 49 RESULT: PREPROCESSING PIPELINE DEFINITION PASSED


In [ ]:
# ================================================================
# CELL 50: FIT AND VALIDATE THE PREPROCESSING TRANSFORMATION
# ================================================================
#
# Purpose:
# Fit preprocessing using TRAINING DATA ONLY, then transform the
# validation and test sets using that already-fitted preprocessor.
#
# This is a critical leakage-control step.
#
# We must NOT do:
#     preprocessor.fit(X_validation)
# or:
#     preprocessor.fit(X_test)
#
# Validation and test data must pass through transformations
# learned exclusively from the training data.
# ================================================================

print("=" * 70)
print("CELL 50: FIT AND VALIDATE THE PREPROCESSING TRANSFORMATION")
print("=" * 70)

# ------------------------------------------------
# 1. Fit preprocessing ONLY on training predictors
# ------------------------------------------------

preprocessor.fit(X_train)

print("\nPREPROCESSOR FIT")
print("-" * 50)
print("Preprocessor fitted using X_train only.")

# ------------------------------------------------
# 2. Transform all three datasets
# ------------------------------------------------

X_train_processed = preprocessor.transform(X_train)

X_validation_processed = preprocessor.transform(X_validation)

X_test_processed = preprocessor.transform(X_test)

print("\nTRANSFORMED DATASET SHAPES")
print("-" * 50)

print(f"X_train_processed:      {X_train_processed.shape}")
print(f"X_validation_processed: {X_validation_processed.shape}")
print(f"X_test_processed:       {X_test_processed.shape}")

# ------------------------------------------------
# 3. Confirm identical feature dimensions
# ------------------------------------------------

same_feature_count = (
    X_train_processed.shape[1]
    == X_validation_processed.shape[1]
    == X_test_processed.shape[1]
)

print("\nFEATURE-DIMENSION CHECK")
print("-" * 50)
print(f"Same transformed feature count: {same_feature_count}")

# ------------------------------------------------
# 4. Check transformed matrices for missing values
# ------------------------------------------------
#
# Depending on the sklearn version and sparse/dense format,
# we handle both possibilities.
# ------------------------------------------------

def count_missing_transformed(matrix):
    """
    Count NaN values in a transformed matrix.

    Works with both sparse and dense matrices.
    """
    if hasattr(matrix, "toarray"):
        matrix_to_check = matrix.toarray()
    else:
        matrix_to_check = matrix

    return np.isnan(matrix_to_check).sum()


train_missing = count_missing_transformed(X_train_processed)
validation_missing = count_missing_transformed(X_validation_processed)
test_missing = count_missing_transformed(X_test_processed)

print("\nTRANSFORMED MISSING-VALUE CHECK")
print("-" * 50)
print(f"Training missing values:   {train_missing}")
print(f"Validation missing values: {validation_missing}")
print(f"Test missing values:       {test_missing}")

# ------------------------------------------------
# 5. Inspect feature names
# ------------------------------------------------

feature_names = preprocessor.get_feature_names_out()

print("\nTRANSFORMED FEATURE INFORMATION")
print("-" * 50)
print(f"Number of transformed features: {len(feature_names)}")

print("\nFirst 20 transformed features:")

for i, feature in enumerate(feature_names[:20], start=1):
    print(f"{i:2d}. {feature}")

if len(feature_names) > 20:
    print(f"... and {len(feature_names) - 20} more features.")

# ------------------------------------------------
# 6. Check preprocessing remains consistent
# ------------------------------------------------
#
# The same fitted preprocessor must produce the same feature
# structure for all datasets.
# ------------------------------------------------

train_feature_names = preprocessor.get_feature_names_out()
validation_feature_names = preprocessor.get_feature_names_out()
test_feature_names = preprocessor.get_feature_names_out()

feature_name_consistency = (
    np.array_equal(train_feature_names, validation_feature_names)
    and
    np.array_equal(train_feature_names, test_feature_names)
)

print("\nFEATURE-NAME CONSISTENCY CHECK")
print("-" * 50)
print(f"Feature structure consistent: {feature_name_consistency}")

# ------------------------------------------------
# 7. Confirm the preprocessor is fitted
# ------------------------------------------------

preprocessor_fitted = hasattr(
    preprocessor,
    "transformers_"
)

print("\nPREPROCESSOR STATUS")
print("-" * 50)
print(f"Preprocessor fitted: {preprocessor_fitted}")

# ------------------------------------------------
# 8. Final validation
# ------------------------------------------------

cell_50_pass = (
    preprocessor_fitted
    and
    same_feature_count
    and
    train_missing == 0
    and
    validation_missing == 0
    and
    test_missing == 0
    and
    len(feature_names) == X_train_processed.shape[1]
    and
    feature_name_consistency
)

print("\n" + "=" * 70)

if cell_50_pass:
    print("CELL 50 RESULT: PREPROCESSING TRANSFORMATION PASSED")
else:
    print("CELL 50 RESULT: REVIEW TRANSFORMATION BEFORE MODELLING")

print("=" * 70)

CELL 50: FIT AND VALIDATE THE PREPROCESSING TRANSFORMATION

PREPROCESSOR FIT
--------------------------------------------------
Preprocessor fitted using X_train only.

TRANSFORMED DATASET SHAPES
--------------------------------------------------
X_train_processed:      (109913, 50)
X_validation_processed: (28190, 50)
X_test_processed:       (36996, 50)

FEATURE-DIMENSION CHECK
--------------------------------------------------
Same transformed feature count: True

TRANSFORMED MISSING-VALUE CHECK
--------------------------------------------------
Training missing values:   0
Validation missing values: 0
Test missing values:       0

TRANSFORMED FEATURE INFORMATION
--------------------------------------------------
Number of transformed features: 50

First 20 transformed features:
 1. numerical__tenure_days
 2. numerical__recency_days
 3. numerical__frequency
 4. numerical__monetary_total
 5. numerical__monetary_average
 6. numerical__support_ticket_count
 7. numerical__satisfaction_ave

In [ ]:
# ================================================================
# CELL 51: BASELINE MODEL — DUMMY CLASSIFIER
# ================================================================
#
# Purpose:
# Establish a simple benchmark before training real ML models.
#
# The DummyClassifier does not learn meaningful customer behaviour.
# With strategy="most_frequent", it always predicts the class that
# is most common in the TRAINING data.
#
# Why is this important?
# A sophisticated model is only useful if it performs better than
# a simple baseline.
#
# IMPORTANT:
# The TEST SET remains untouched.
# We evaluate the baseline on VALIDATION data only.
# ================================================================

print("=" * 70)
print("CELL 51: BASELINE MODEL — DUMMY CLASSIFIER")
print("=" * 70)

# ------------------------------------------------
# 1. Import baseline model and metrics
# ------------------------------------------------

from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# ------------------------------------------------
# 2. Create baseline classifier
# ------------------------------------------------

baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# ------------------------------------------------
# 3. Fit ONLY on training data
# ------------------------------------------------

baseline_model.fit(
    X_train_processed,
    y_train
)

print("\nBASELINE FIT")
print("-" * 50)
print("DummyClassifier fitted on training data.")
print("Strategy: most_frequent")

# ------------------------------------------------
# 4. Generate validation predictions
# ------------------------------------------------

baseline_validation_pred = baseline_model.predict(
    X_validation_processed
)

baseline_validation_proba = baseline_model.predict_proba(
    X_validation_processed
)[:, 1]

# ------------------------------------------------
# 5. Calculate validation metrics
# ------------------------------------------------

baseline_accuracy = accuracy_score(
    y_validation,
    baseline_validation_pred
)

baseline_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    baseline_validation_pred
)

baseline_precision = precision_score(
    y_validation,
    baseline_validation_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_validation,
    baseline_validation_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_validation,
    baseline_validation_pred,
    zero_division=0
)

baseline_roc_auc = roc_auc_score(
    y_validation,
    baseline_validation_proba
)

baseline_pr_auc = average_precision_score(
    y_validation,
    baseline_validation_proba
)

baseline_cm = confusion_matrix(
    y_validation,
    baseline_validation_pred
)

# ------------------------------------------------
# 6. Display results
# ------------------------------------------------

print("\nVALIDATION PERFORMANCE")
print("-" * 50)

print(f"Accuracy:          {baseline_accuracy:.4f}")
print(f"Balanced Accuracy: {baseline_balanced_accuracy:.4f}")
print(f"Precision:         {baseline_precision:.4f}")
print(f"Recall:            {baseline_recall:.4f}")
print(f"F1 Score:          {baseline_f1:.4f}")
print(f"ROC-AUC:           {baseline_roc_auc:.4f}")
print(f"PR-AUC:            {baseline_pr_auc:.4f}")

print("\nCONFUSION MATRIX")
print("-" * 50)
print("Rows = Actual | Columns = Predicted")
print(cm := baseline_cm)

# ------------------------------------------------
# 7. Confirm baseline prediction behaviour
# ------------------------------------------------

unique_predictions = np.unique(
    baseline_validation_pred
)

print("\nPREDICTION BEHAVIOUR")
print("-" * 50)
print(
    f"Unique predicted classes: "
    f"{unique_predictions.tolist()}"
)

print(
    f"Training majority class: "
    f"{baseline_model.classes_[np.argmax(baseline_model.class_prior_)]}"
)

# ------------------------------------------------
# 8. Final validation
# ------------------------------------------------

baseline_pass = (
    len(baseline_validation_pred) == len(y_validation)
    and
    len(baseline_validation_proba) == len(y_validation)
    and
    set(unique_predictions).issubset({0, 1})
    and
    np.isfinite(baseline_accuracy)
    and
    np.isfinite(baseline_balanced_accuracy)
    and
    np.isfinite(baseline_f1)
    and
    np.isfinite(baseline_roc_auc)
    and
    np.isfinite(baseline_pr_auc)
)

print("\n" + "=" * 70)

if baseline_pass:
    print("CELL 51 RESULT: BASELINE MODEL PASSED")
else:
    print("CELL 51 RESULT: REVIEW BASELINE MODEL")

print("=" * 70)

CELL 51: BASELINE MODEL — DUMMY CLASSIFIER

BASELINE FIT
--------------------------------------------------
DummyClassifier fitted on training data.
Strategy: most_frequent

VALIDATION PERFORMANCE
--------------------------------------------------
Accuracy:          0.6013
Balanced Accuracy: 0.5000
Precision:         0.6013
Recall:            1.0000
F1 Score:          0.7510
ROC-AUC:           0.5000
PR-AUC:            0.6013

CONFUSION MATRIX
--------------------------------------------------
Rows = Actual | Columns = Predicted
[[    0 11239]
 [    0 16951]]

PREDICTION BEHAVIOUR
--------------------------------------------------
Unique predicted classes: [1]
Training majority class: 1

CELL 51 RESULT: BASELINE MODEL PASSED


In [ ]:
# ================================================================
# CELL 52: LOGISTIC REGRESSION BASELINE MODEL
# ================================================================
#
# Purpose:
# Train the first genuine predictive model for future customer
# churn.
#
# Logistic Regression is useful as an interpretable baseline
# because it estimates the probability of class 1 (future churn)
# and provides coefficients that can later be examined.
#
# IMPORTANT:
# The preprocessing is placed inside the model pipeline.
# This ensures that the complete workflow can be reproduced
# without accidentally fitting preprocessing on validation/test data.
#
# TEST DATA REMAINS UNTOUCHED.
# ================================================================

print("=" * 70)
print("CELL 52: LOGISTIC REGRESSION BASELINE MODEL")
print("=" * 70)

# ------------------------------------------------
# 1. Import Logistic Regression
# ------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# ------------------------------------------------
# 2. Create the complete modelling pipeline
# ------------------------------------------------
#
# We use the already-defined "preprocessor".
#
# The classifier receives the 50 transformed features created
# from the original 13 predictors.
# ------------------------------------------------

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("\nMODEL CONFIGURATION")
print("-" * 50)
print("Model: Logistic Regression")
print("Maximum iterations: 1000")
print("Random state: 42")
print("Class weighting: None")
print("Test set: untouched")

# ------------------------------------------------
# 3. Fit model on training data
# ------------------------------------------------

logistic_model.fit(
    X_train,
    y_train
)

print("\nMODEL FIT")
print("-" * 50)
print("Logistic Regression fitted on training data.")

# ------------------------------------------------
# 4. Generate validation predictions
# ------------------------------------------------

logistic_validation_pred = logistic_model.predict(
    X_validation
)

logistic_validation_proba = logistic_model.predict_proba(
    X_validation
)[:, 1]

# ------------------------------------------------
# 5. Calculate validation metrics
# ------------------------------------------------

logistic_accuracy = accuracy_score(
    y_validation,
    logistic_validation_pred
)

logistic_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    logistic_validation_pred
)

logistic_precision = precision_score(
    y_validation,
    logistic_validation_pred,
    zero_division=0
)

logistic_recall = recall_score(
    y_validation,
    logistic_validation_pred,
    zero_division=0
)

logistic_f1 = f1_score(
    y_validation,
    logistic_validation_pred,
    zero_division=0
)

logistic_roc_auc = roc_auc_score(
    y_validation,
    logistic_validation_proba
)

logistic_pr_auc = average_precision_score(
    y_validation,
    logistic_validation_proba
)

logistic_cm = confusion_matrix(
    y_validation,
    logistic_validation_pred
)

# ------------------------------------------------
# 6. Display validation performance
# ------------------------------------------------

print("\nVALIDATION PERFORMANCE")
print("-" * 50)

print(f"Accuracy:          {logistic_accuracy:.4f}")
print(f"Balanced Accuracy: {logistic_balanced_accuracy:.4f}")
print(f"Precision:         {logistic_precision:.4f}")
print(f"Recall:            {logistic_recall:.4f}")
print(f"F1 Score:          {logistic_f1:.4f}")
print(f"ROC-AUC:           {logistic_roc_auc:.4f}")
print(f"PR-AUC:            {logistic_pr_auc:.4f}")

# ------------------------------------------------
# 7. Confusion matrix
# ------------------------------------------------

print("\nCONFUSION MATRIX")
print("-" * 50)
print("Rows = Actual | Columns = Predicted")
print(logistic_cm)

# ------------------------------------------------
# 8. Prediction class distribution
# ------------------------------------------------

print("\nPREDICTED CLASS DISTRIBUTION")
print("-" * 50)

print(
    pd.Series(
        logistic_validation_pred
    ).value_counts().sort_index()
)

# ------------------------------------------------
# 9. Compare against the dummy baseline
# ------------------------------------------------

print("\nCOMPARISON WITH DUMMY BASELINE")
print("-" * 50)

print(
    f"Baseline ROC-AUC:   {baseline_roc_auc:.4f}"
)

print(
    f"Logistic ROC-AUC:   {logistic_roc_auc:.4f}"
)

print(
    f"Baseline PR-AUC:    {baseline_pr_auc:.4f}"
)

print(
    f"Logistic PR-AUC:    {logistic_pr_auc:.4f}"
)

roc_auc_improvement = (
    logistic_roc_auc - baseline_roc_auc
)

pr_auc_improvement = (
    logistic_pr_auc - baseline_pr_auc
)

print(
    f"\nROC-AUC improvement: {roc_auc_improvement:+.4f}"
)

print(
    f"PR-AUC improvement:  {pr_auc_improvement:+.4f}"
)

# ------------------------------------------------
# 10. Final validation
# ------------------------------------------------

logistic_pass = (
    len(logistic_validation_pred) == len(y_validation)
    and
    len(logistic_validation_proba) == len(y_validation)
    and
    np.isfinite(logistic_accuracy)
    and
    np.isfinite(logistic_balanced_accuracy)
    and
    np.isfinite(logistic_precision)
    and
    np.isfinite(logistic_recall)
    and
    np.isfinite(logistic_f1)
    and
    np.isfinite(logistic_roc_auc)
    and
    np.isfinite(logistic_pr_auc)
    and
    logistic_roc_auc >= baseline_roc_auc
)

print("\n" + "=" * 70)

if logistic_pass:
    print("CELL 52 RESULT: LOGISTIC REGRESSION MODEL PASSED")
else:
    print("CELL 52 RESULT: REVIEW LOGISTIC REGRESSION RESULTS")

print("=" * 70)

CELL 52: LOGISTIC REGRESSION BASELINE MODEL

MODEL CONFIGURATION
--------------------------------------------------
Model: Logistic Regression
Maximum iterations: 1000
Random state: 42
Class weighting: None
Test set: untouched

MODEL FIT
--------------------------------------------------
Logistic Regression fitted on training data.

VALIDATION PERFORMANCE
--------------------------------------------------
Accuracy:          0.6150
Balanced Accuracy: 0.5499
Precision:         0.6301
Recall:            0.8713
F1 Score:          0.7313
ROC-AUC:           0.5952
PR-AUC:            0.6583

CONFUSION MATRIX
--------------------------------------------------
Rows = Actual | Columns = Predicted
[[ 2569  8670]
 [ 2182 14769]]

PREDICTED CLASS DISTRIBUTION
--------------------------------------------------
0     4751
1    23439
Name: count, dtype: int64

COMPARISON WITH DUMMY BASELINE
--------------------------------------------------
Baseline ROC-AUC:   0.5000
Logistic ROC-AUC:   0.5952
Baselin

In [ ]:
# ================================================================
# CELL 53: RANDOM FOREST MODEL
# ================================================================
#
# Purpose:
# Train a nonlinear tree-based model and compare it with the
# Logistic Regression model.
#
# Random Forest can capture nonlinear relationships and feature
# interactions that Logistic Regression may not capture directly.
#
# IMPORTANT:
# - Training data is used for fitting.
# - Validation data is used for evaluation.
# - Test data remains untouched.
# - The preprocessing remains inside the pipeline.
# ================================================================

print("=" * 70)
print("CELL 53: RANDOM FOREST MODEL")
print("=" * 70)

# ------------------------------------------------
# 1. Import Random Forest
# ------------------------------------------------

from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------
# 2. Create Random Forest pipeline
# ------------------------------------------------
#
# n_estimators=200 gives the forest 200 decision trees.
#
# random_state=42 ensures reproducibility.
#
# n_jobs=-1 allows sklearn to use available CPU cores.
#
# We initially use class_weight=None because the training
# target is not extremely imbalanced (about 61% churn).
#
# We can investigate class weighting later if necessary.
# ------------------------------------------------

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
                class_weight=None
            )
        )
    ]
)

print("\nMODEL CONFIGURATION")
print("-" * 50)
print("Model: Random Forest Classifier")
print("Number of trees: 200")
print("Random state: 42")
print("Class weighting: None")
print("Parallel processing: Enabled")
print("Test set: untouched")

# ------------------------------------------------
# 3. Fit on training data
# ------------------------------------------------

random_forest_model.fit(
    X_train,
    y_train
)

print("\nMODEL FIT")
print("-" * 50)
print("Random Forest fitted on training data.")

# ------------------------------------------------
# 4. Generate validation predictions
# ------------------------------------------------

rf_validation_pred = random_forest_model.predict(
    X_validation
)

rf_validation_proba = random_forest_model.predict_proba(
    X_validation
)[:, 1]

# ------------------------------------------------
# 5. Calculate validation metrics
# ------------------------------------------------

rf_accuracy = accuracy_score(
    y_validation,
    rf_validation_pred
)

rf_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    rf_validation_pred
)

rf_precision = precision_score(
    y_validation,
    rf_validation_pred,
    zero_division=0
)

rf_recall = recall_score(
    y_validation,
    rf_validation_pred,
    zero_division=0
)

rf_f1 = f1_score(
    y_validation,
    rf_validation_pred,
    zero_division=0
)

rf_roc_auc = roc_auc_score(
    y_validation,
    rf_validation_proba
)

rf_pr_auc = average_precision_score(
    y_validation,
    rf_validation_proba
)

rf_cm = confusion_matrix(
    y_validation,
    rf_validation_pred
)

# ------------------------------------------------
# 6. Display validation performance
# ------------------------------------------------

print("\nVALIDATION PERFORMANCE")
print("-" * 50)

print(f"Accuracy:          {rf_accuracy:.4f}")
print(f"Balanced Accuracy: {rf_balanced_accuracy:.4f}")
print(f"Precision:         {rf_precision:.4f}")
print(f"Recall:            {rf_recall:.4f}")
print(f"F1 Score:          {rf_f1:.4f}")
print(f"ROC-AUC:           {rf_roc_auc:.4f}")
print(f"PR-AUC:            {rf_pr_auc:.4f}")

# ------------------------------------------------
# 7. Confusion matrix
# ------------------------------------------------

print("\nCONFUSION MATRIX")
print("-" * 50)
print("Rows = Actual | Columns = Predicted")
print(rf_cm)

# ------------------------------------------------
# 8. Predicted class distribution
# ------------------------------------------------

print("\nPREDICTED CLASS DISTRIBUTION")
print("-" * 50)

print(
    pd.Series(
        rf_validation_pred
    ).value_counts().sort_index()
)

# ------------------------------------------------
# 9. Compare all current models
# ------------------------------------------------

print("\nMODEL COMPARISON")
print("-" * 50)

print(
    f"{'Model':<25}"
    f"{'ROC-AUC':>10}"
    f"{'PR-AUC':>10}"
    f"{'F1':>10}"
)

print("-" * 55)

print(
    f"{'Dummy Baseline':<25}"
    f"{baseline_roc_auc:>10.4f}"
    f"{baseline_pr_auc:>10.4f}"
    f"{baseline_f1:>10.4f}"
)

print(
    f"{'Logistic Regression':<25}"
    f"{logistic_roc_auc:>10.4f}"
    f"{logistic_pr_auc:>10.4f}"
    f"{logistic_f1:>10.4f}"
)

print(
    f"{'Random Forest':<25}"
    f"{rf_roc_auc:>10.4f}"
    f"{rf_pr_auc:>10.4f}"
    f"{rf_f1:>10.4f}"
)

# ------------------------------------------------
# 10. Compare Random Forest against Logistic Regression
# ------------------------------------------------

print("\nRANDOM FOREST VS LOGISTIC REGRESSION")
print("-" * 50)

print(
    f"ROC-AUC difference: "
    f"{rf_roc_auc - logistic_roc_auc:+.4f}"
)

print(
    f"PR-AUC difference:  "
    f"{rf_pr_auc - logistic_pr_auc:+.4f}"
)

# ------------------------------------------------
# 11. Final validation
# ------------------------------------------------

rf_pass = (
    len(rf_validation_pred) == len(y_validation)
    and
    len(rf_validation_proba) == len(y_validation)
    and
    np.isfinite(rf_accuracy)
    and
    np.isfinite(rf_balanced_accuracy)
    and
    np.isfinite(rf_precision)
    and
    np.isfinite(rf_recall)
    and
    np.isfinite(rf_f1)
    and
    np.isfinite(rf_roc_auc)
    and
    np.isfinite(rf_pr_auc)
)

print("\n" + "=" * 70)

if rf_pass:
    print("CELL 53 RESULT: RANDOM FOREST MODEL PASSED")
else:
    print("CELL 53 RESULT: REVIEW RANDOM FOREST RESULTS")

print("=" * 70)

CELL 53: RANDOM FOREST MODEL

MODEL CONFIGURATION
--------------------------------------------------
Model: Random Forest Classifier
Number of trees: 200
Random state: 42
Class weighting: None
Parallel processing: Enabled
Test set: untouched

MODEL FIT
--------------------------------------------------
Random Forest fitted on training data.

VALIDATION PERFORMANCE
--------------------------------------------------
Accuracy:          0.7233
Balanced Accuracy: 0.6865
Precision:         0.7257
Recall:            0.8680
F1 Score:          0.7905
ROC-AUC:           0.7382
PR-AUC:            0.7473

CONFUSION MATRIX
--------------------------------------------------
Rows = Actual | Columns = Predicted
[[ 5677  5562]
 [ 2238 14713]]

PREDICTED CLASS DISTRIBUTION
--------------------------------------------------
0     7915
1    20275
Name: count, dtype: int64

MODEL COMPARISON
--------------------------------------------------
Model                       ROC-AUC    PR-AUC        F1
----------

In [ ]:
# ================================================================
# CELL 54: GRADIENT BOOSTING MODEL
# ================================================================

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 70)
print("CELL 54: GRADIENT BOOSTING MODEL")
print("=" * 70)

# ------------------------------------------------
# 1. MODEL CONFIGURATION
# ------------------------------------------------

print("\nMODEL CONFIGURATION")
print("-" * 50)
print("Model: Gradient Boosting Classifier")
print("Number of boosting stages: 100")
print("Learning rate: 0.10")
print("Maximum tree depth: 3")
print("Random state: 42")
print("Test set: untouched")

# ------------------------------------------------
# 2. CREATE MODEL PIPELINE
# ------------------------------------------------

gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingClassifier(
                n_estimators=100,
                learning_rate=0.10,
                max_depth=3,
                random_state=42
            )
        )
    ]
)

# ------------------------------------------------
# 3. FIT MODEL
# ------------------------------------------------

print("\nMODEL FIT")
print("-" * 50)

gb_pipeline.fit(X_train, y_train)

print("Gradient Boosting fitted on training data.")

# ------------------------------------------------
# 4. VALIDATION PREDICTIONS
# ------------------------------------------------

gb_val_pred = gb_pipeline.predict(X_val)
gb_val_proba = gb_pipeline.predict_proba(X_val)[:, 1]

# ------------------------------------------------
# 5. VALIDATION METRICS
# ------------------------------------------------

gb_accuracy = accuracy_score(y_val, gb_val_pred)
gb_balanced_accuracy = balanced_accuracy_score(y_val, gb_val_pred)
gb_precision = precision_score(y_val, gb_val_pred, zero_division=0)
gb_recall = recall_score(y_val, gb_val_pred, zero_division=0)
gb_f1 = f1_score(y_val, gb_val_pred, zero_division=0)
gb_roc_auc = roc_auc_score(y_val, gb_val_proba)
gb_pr_auc = average_precision_score(y_val, gb_val_proba)

print("\nVALIDATION PERFORMANCE")
print("-" * 50)
print(f"Accuracy:          {gb_accuracy:.4f}")
print(f"Balanced Accuracy: {gb_balanced_accuracy:.4f}")
print(f"Precision:         {gb_precision:.4f}")
print(f"Recall:            {gb_recall:.4f}")
print(f"F1 Score:          {gb_f1:.4f}")
print(f"ROC-AUC:           {gb_roc_auc:.4f}")
print(f"PR-AUC:            {gb_pr_auc:.4f}")

# ------------------------------------------------
# 6. CONFUSION MATRIX
# ------------------------------------------------

gb_cm = confusion_matrix(y_val, gb_val_pred)

print("\nCONFUSION MATRIX")
print("-" * 50)
print("Rows = Actual | Columns = Predicted")
print(gb_cm)

# ------------------------------------------------
# 7. PREDICTED CLASS DISTRIBUTION
# ------------------------------------------------

print("\nPREDICTED CLASS DISTRIBUTION")
print("-" * 50)
print(
    pd.Series(gb_val_pred)
    .value_counts()
    .sort_index()
)

# ------------------------------------------------
# 8. UPDATED MODEL COMPARISON
# ------------------------------------------------

comparison_df = pd.DataFrame({
    "Model": [
        "Dummy Baseline",
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "ROC-AUC": [
        0.5000,
        0.5952,
        0.7382,
        gb_roc_auc
    ],
    "PR-AUC": [
        0.6013,
        0.6583,
        0.7473,
        gb_pr_auc
    ],
    "F1": [
        0.7510,
        0.7313,
        0.7905,
        gb_f1
    ]
})

print("\nMODEL COMPARISON")
print("-" * 50)
print(
    comparison_df.to_string(
        index=False,
        formatters={
            "ROC-AUC": "{:.4f}".format,
            "PR-AUC": "{:.4f}".format,
            "F1": "{:.4f}".format
        }
    )
)

print("\n" + "=" * 70)
print("CELL 54 RESULT: GRADIENT BOOSTING MODEL PASSED")
print("=" * 70)

In [ ]:
# ================================================================
# CELL 55: MODEL COMPARISON & CLASS BALANCE AUDIT
# ================================================================

print("=" * 70)
print("CELL 55: MODEL COMPARISON & CLASS BALANCE AUDIT")
print("=" * 70)

# ------------------------------------------------
# 1. CLASS DISTRIBUTION
# ------------------------------------------------

print("\nTRAINING CLASS DISTRIBUTION")
print("-" * 50)

train_class_counts = y_train.value_counts().sort_index()
train_class_pct = (
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

for class_value in train_class_counts.index:
    print(
        f"Class {class_value}: "
        f"{train_class_counts[class_value]:,} "
        f"({train_class_pct[class_value]:.2f}%)"
    )

# Calculate minority/majority ratio
majority_count = train_class_counts.max()
minority_count = train_class_counts.min()

imbalance_ratio = majority_count / minority_count

print(f"\nMajority / minority ratio: {imbalance_ratio:.2f}:1")

# ------------------------------------------------
# 2. VALIDATION CLASS DISTRIBUTION
# ------------------------------------------------

print("\nVALIDATION CLASS DISTRIBUTION")
print("-" * 50)

val_class_counts = y_val.value_counts().sort_index()
val_class_pct = (
    y_val.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

for class_value in val_class_counts.index:
    print(
        f"Class {class_value}: "
        f"{val_class_counts[class_value]:,} "
        f"({val_class_pct[class_value]:.2f}%)"
    )

# ------------------------------------------------
# 3. MODEL COMPARISON TABLE
# ------------------------------------------------

model_comparison = pd.DataFrame({
    "Model": [
        "Dummy Baseline",
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "ROC-AUC": [
        0.5000,
        0.5952,
        0.7382,
        0.6121
    ],
    "PR-AUC": [
        0.6013,
        0.6583,
        0.7473,
        0.6598
    ],
    "F1": [
        0.7510,
        0.7313,
        0.7905,
        0.7501
    ],
    "Recall": [
        1.0000,
        0.8713,
        0.8680,
        0.9276
    ],
    "Precision": [
        0.6013,
        0.6301,
        0.7257,
        0.6297
    ]
})

print("\nMODEL COMPARISON")
print("-" * 50)

print(
    model_comparison.to_string(
        index=False,
        formatters={
            "ROC-AUC": "{:.4f}".format,
            "PR-AUC": "{:.4f}".format,
            "F1": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "Precision": "{:.4f}".format
        }
    )
)

# ------------------------------------------------
# 4. BEST CURRENT MODEL BY KEY METRICS
# ------------------------------------------------

print("\nCURRENT VALIDATION LEADERS")
print("-" * 50)

best_roc_model = model_comparison.loc[
    model_comparison["ROC-AUC"].idxmax(),
    "Model"
]

best_pr_model = model_comparison.loc[
    model_comparison["PR-AUC"].idxmax(),
    "Model"
]

best_f1_model = model_comparison.loc[
    model_comparison["F1"].idxmax(),
    "Model"
]

print(f"Highest ROC-AUC: {best_roc_model}")
print(f"Highest PR-AUC:  {best_pr_model}")
print(f"Highest F1:      {best_f1_model}")

# ------------------------------------------------
# 5. RUBRIC-RELEVANT INTERPRETATION
# ------------------------------------------------

print("\nMODELLING DECISION NOTES")
print("-" * 50)

if imbalance_ratio < 2:
    print(
        "Class imbalance is moderate rather than extreme "
        f"({imbalance_ratio:.2f}:1)."
    )
else:
    print(
        "Class imbalance is substantial "
        f"({imbalance_ratio:.2f}:1)."
    )

print(
    "Accuracy will not be used as the sole model-selection metric."
)

print(
    "ROC-AUC, PR-AUC, balanced accuracy, precision, recall and F1 "
    "will be considered together."
)

print(
    "Random Forest currently provides the strongest validation "
    "ROC-AUC, PR-AUC and F1 among the predictive models."
)

print(
    "The test set remains untouched."
)

print("\n" + "=" * 70)
print("CELL 55 RESULT: MODEL COMPARISON & CLASS BALANCE AUDIT COMPLETE")
print("=" * 70)

CELL 55: MODEL COMPARISON & CLASS BALANCE AUDIT

TRAINING CLASS DISTRIBUTION
--------------------------------------------------
Class 0: 43,044 (39.16%)
Class 1: 66,869 (60.84%)

Majority / minority ratio: 1.55:1

VALIDATION CLASS DISTRIBUTION
--------------------------------------------------
Class 0: 11,239 (39.87%)
Class 1: 16,951 (60.13%)

MODEL COMPARISON
--------------------------------------------------
              Model ROC-AUC PR-AUC     F1 Recall Precision
     Dummy Baseline  0.5000 0.6013 0.7510 1.0000    0.6013
Logistic Regression  0.5952 0.6583 0.7313 0.8713    0.6301
      Random Forest  0.7382 0.7473 0.7905 0.8680    0.7257
  Gradient Boosting  0.6121 0.6598 0.7501 0.9276    0.6297

CURRENT VALIDATION LEADERS
--------------------------------------------------
Highest ROC-AUC: Random Forest
Highest PR-AUC:  Random Forest
Highest F1:      Random Forest

MODELLING DECISION NOTES
--------------------------------------------------
Class imbalance is moderate rather than ex

In [ ]:
# ================================================================
# CELL 56: RANDOM FOREST HYPERPARAMETER TUNING
# ================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint

print("=" * 70)
print("CELL 56: RANDOM FOREST HYPERPARAMETER TUNING")
print("=" * 70)

# ------------------------------------------------
# 1. SORT TRAINING DATA CHRONOLOGICALLY
# ------------------------------------------------
# TimeSeriesSplit requires the observations to be ordered
# chronologically so that validation folds do not precede
# their corresponding training folds.

train_order = ml_features["prediction_date"].loc[
    ml_features["prediction_date"].isin(
        ml_features.loc[
            ml_features["prediction_date"] <= pd.Timestamp("2024-12-01"),
            "prediction_date"
        ].unique()
    )
].sort_values().index

X_train_tuning = X_train.loc[train_order.intersection(X_train.index)].copy()
y_train_tuning = y_train.loc[train_order.intersection(y_train.index)].copy()

# ------------------------------------------------
# 2. DEFINE BASE PIPELINE
# ------------------------------------------------

rf_tuning_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

# ------------------------------------------------
# 3. DEFINE SEARCH SPACE
# ------------------------------------------------

param_dist = {
    "model__n_estimators": randint(150, 401),
    "model__max_depth": [None, 10, 15, 20, 25, 30],
    "model__min_samples_split": randint(2, 11),
    "model__min_samples_leaf": randint(1, 6),
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": [None, "balanced"]
}

# ------------------------------------------------
# 4. TIME-AWARE CROSS-VALIDATION
# ------------------------------------------------

time_cv = TimeSeriesSplit(n_splits=3)

# ------------------------------------------------
# 5. RANDOMIZED SEARCH
# ------------------------------------------------

rf_random_search = RandomizedSearchCV(
    estimator=rf_tuning_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=time_cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("\nTUNING CONFIGURATION")
print("-" * 50)

print(f"Training observations: {len(X_train_tuning):,}")
print("Cross-validation: TimeSeriesSplit")
print("Number of CV folds: 3")
print("Randomized search iterations: 10")
print("Optimization metric: ROC-AUC")
print("Class weighting tested: None and balanced")
print("Test set: untouched")

# ------------------------------------------------
# 6. FIT SEARCH
# ------------------------------------------------

print("\nSTARTING RANDOMIZED SEARCH")
print("-" * 50)

rf_random_search.fit(
    X_train_tuning,
    y_train_tuning
)

print("\nRANDOMIZED SEARCH COMPLETE")

# ------------------------------------------------
# 7. BEST PARAMETERS
# ------------------------------------------------

print("\nBEST PARAMETERS")
print("-" * 50)

for parameter, value in rf_random_search.best_params_.items():
    print(f"{parameter}: {value}")

# ------------------------------------------------
# 8. BEST CROSS-VALIDATION SCORE
# ------------------------------------------------

print("\nBEST CROSS-VALIDATION PERFORMANCE")
print("-" * 50)

print(
    f"Best mean CV ROC-AUC: "
    f"{rf_random_search.best_score_:.4f}"
)

print("\n" + "=" * 70)
print("CELL 56 RESULT: RANDOM FOREST TUNING COMPLETE")
print("=" * 70)

CELL 56: RANDOM FOREST HYPERPARAMETER TUNING

TUNING CONFIGURATION
--------------------------------------------------
Training observations: 109,913
Cross-validation: TimeSeriesSplit
Number of CV folds: 3
Randomized search iterations: 10
Optimization metric: ROC-AUC
Class weighting tested: None and balanced
Test set: untouched

STARTING RANDOMIZED SEARCH
--------------------------------------------------
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.




RANDOMIZED SEARCH COMPLETE

BEST PARAMETERS
--------------------------------------------------
model__class_weight: None
model__max_depth: 20
model__max_features: log2
model__min_samples_leaf: 1
model__min_samples_split: 4
model__n_estimators: 378

BEST CROSS-VALIDATION PERFORMANCE
--------------------------------------------------
Best mean CV ROC-AUC: 0.6963

CELL 56 RESULT: RANDOM FOREST TUNING COMPLETE


In [ ]:
# ======================================================================
# CELL 57: EVALUATE TUNED RANDOM FOREST ON VALIDATION SET
# ======================================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 70)
print("CELL 57: TUNED RANDOM FOREST VALIDATION EVALUATION")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. Get the best Random Forest pipeline from Cell 56
# ----------------------------------------------------------------------
# RandomizedSearchCV has already identified the best combination of
# hyperparameters. We reuse that fitted best estimator here.
tuned_rf_pipeline = rf_random_search.best_estimator_

print("\nBEST TUNED PARAMETERS")
print("-" * 50)

best_params = rf_random_search.best_params_

for parameter, value in best_params.items():
    print(f"{parameter}: {value}")


# ----------------------------------------------------------------------
# 2. Generate predictions for the validation set
# ----------------------------------------------------------------------
# IMPORTANT:
# X_val and y_val represent the 2025-01-01 to 2025-06-01 period.
#
# The test set is deliberately NOT used here.
y_val_pred_tuned_rf = tuned_rf_pipeline.predict(X_val)

# Probability of class 1 (future churn)
y_val_prob_tuned_rf = tuned_rf_pipeline.predict_proba(X_val)[:, 1]


# ----------------------------------------------------------------------
# 3. Calculate validation metrics
# ----------------------------------------------------------------------
tuned_rf_accuracy = accuracy_score(y_val, y_val_pred_tuned_rf)

tuned_rf_balanced_accuracy = balanced_accuracy_score(
    y_val,
    y_val_pred_tuned_rf
)

tuned_rf_precision = precision_score(
    y_val,
    y_val_pred_tuned_rf,
    zero_division=0
)

tuned_rf_recall = recall_score(
    y_val,
    y_val_pred_tuned_rf,
    zero_division=0
)

tuned_rf_f1 = f1_score(
    y_val,
    y_val_pred_tuned_rf,
    zero_division=0
)

tuned_rf_roc_auc = roc_auc_score(
    y_val,
    y_val_prob_tuned_rf
)

tuned_rf_pr_auc = average_precision_score(
    y_val,
    y_val_prob_tuned_rf
)

tuned_rf_cm = confusion_matrix(
    y_val,
    y_val_pred_tuned_rf
)


# ----------------------------------------------------------------------
# 4. Display results
# ----------------------------------------------------------------------
print("\nTUNED RANDOM FOREST VALIDATION PERFORMANCE")
print("-" * 50)

print(f"Accuracy:            {tuned_rf_accuracy:.4f}")
print(f"Balanced Accuracy:   {tuned_rf_balanced_accuracy:.4f}")
print(f"Precision:            {tuned_rf_precision:.4f}")
print(f"Recall:               {tuned_rf_recall:.4f}")
print(f"F1 Score:             {tuned_rf_f1:.4f}")
print(f"ROC-AUC:              {tuned_rf_roc_auc:.4f}")
print(f"PR-AUC:               {tuned_rf_pr_auc:.4f}")

print("\nCONFUSION MATRIX")
print("-" * 50)
print(tuned_rf_cm)

print("\n" + "=" * 70)
print("CELL 57 RESULT: TUNED RANDOM FOREST VALIDATION EVALUATION COMPLETE")
print("=" * 70)

CELL 57: TUNED RANDOM FOREST VALIDATION EVALUATION

BEST TUNED PARAMETERS
--------------------------------------------------
model__class_weight: None
model__max_depth: 20
model__max_features: log2
model__min_samples_leaf: 1
model__min_samples_split: 4
model__n_estimators: 378

TUNED RANDOM FOREST VALIDATION PERFORMANCE
--------------------------------------------------
Accuracy:            0.6828
Balanced Accuracy:   0.6280
Precision:            0.6784
Recall:               0.8985
F1 Score:             0.7731
ROC-AUC:              0.7087
PR-AUC:               0.7366

CONFUSION MATRIX
--------------------------------------------------
[[ 4019  7220]
 [ 1721 15230]]

CELL 57 RESULT: TUNED RANDOM FOREST VALIDATION EVALUATION COMPLETE


In [ ]:
# ======================================================================
# CELL 58: CONSOLIDATED MODEL COMPARISON
# ======================================================================

print("=" * 70)
print("CELL 58: MODEL COMPARISON ON VALIDATION SET")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. Create a comparison table using the validation results already
#    calculated for each model.
#
#    IMPORTANT:
#    The test set is still untouched.
# ----------------------------------------------------------------------

model_comparison = pd.DataFrame({
    "Model": [
        "Dummy Baseline",
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting",
        "Tuned Random Forest"
    ],

    "Accuracy": [
        0.6013,
        0.6150,
        0.7233,
        0.6284,
        tuned_rf_accuracy
    ],

    "Balanced Accuracy": [
        0.5000,
        0.5499,
        0.6865,
        0.5524,
        tuned_rf_balanced_accuracy
    ],

    "Precision": [
        0.6013,
        0.6301,
        0.7257,
        0.6297,
        tuned_rf_precision
    ],

    "Recall": [
        1.0000,
        0.8713,
        0.8680,
        0.9276,
        tuned_rf_recall
    ],

    "F1 Score": [
        0.7510,
        0.7313,
        0.7905,
        0.7501,
        tuned_rf_f1
    ],

    "ROC-AUC": [
        0.5000,
        0.5952,
        0.7382,
        0.6121,
        tuned_rf_roc_auc
    ],

    "PR-AUC": [
        0.6013,
        0.6583,
        0.7473,
        0.6598,
        tuned_rf_pr_auc
    ]
})

# ----------------------------------------------------------------------
# 2. Display the comparison table
# ----------------------------------------------------------------------

print("\nVALIDATION PERFORMANCE COMPARISON")
print("-" * 70)

display(
    model_comparison.style.format({
        "Accuracy": "{:.4f}",
        "Balanced Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}"
    })
)

# ----------------------------------------------------------------------
# 3. Identify the highest value for each important metric
# ----------------------------------------------------------------------

print("\nBEST VALIDATION VALUE BY METRIC")
print("-" * 70)

for metric in [
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "PR-AUC"
]:
    best_index = model_comparison[metric].idxmax()
    best_model = model_comparison.loc[best_index, "Model"]
    best_value = model_comparison.loc[best_index, metric]

    print(f"{metric:<20}: {best_model} ({best_value:.4f})")

print("\n" + "=" * 70)
print("CELL 58 RESULT: MODEL COMPARISON COMPLETE")
print("=" * 70)

CELL 58: MODEL COMPARISON ON VALIDATION SET

VALIDATION PERFORMANCE COMPARISON
----------------------------------------------------------------------


,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
0,Dummy Baseline,0.6013,0.5000,0.6013,1.0000,0.7510,0.5000,0.6013
1,Logistic Regression,0.6150,0.5499,0.6301,0.8713,0.7313,0.5952,0.6583
2,Random Forest,0.7233,0.6865,0.7257,0.8680,0.7905,0.7382,0.7473
3,Gradient Boosting,0.6284,0.5524,0.6297,0.9276,0.7501,0.6121,0.6598
4,Tuned Random Forest,0.6828,0.6280,0.6784,0.8985,0.7731,0.7087,0.7366



BEST VALIDATION VALUE BY METRIC
----------------------------------------------------------------------
Balanced Accuracy   : Random Forest (0.6865)
Precision           : Random Forest (0.7257)
Recall              : Dummy Baseline (1.0000)
F1 Score            : Random Forest (0.7905)
ROC-AUC             : Random Forest (0.7382)
PR-AUC              : Random Forest (0.7473)

CELL 58 RESULT: MODEL COMPARISON COMPLETE
